# Geo-Insight: Which Crises Are Most Overlooked?

**Working prototype notebook — UNOCHA / Databricks Hackathon submission**

This notebook is the **reproducible analytical pipeline** behind the Geo-Insight submission. It takes publicly available UN OCHA datasets (HRP, FTS, HNO, INFORM Severity Index, population) and produces a tier-classified ranking of humanitarian response plans by their mismatch between documented need and available funding, plus a multi-year structural-neglect signal and a geospatial output.

## Headline findings

- **The Syria 3RP (Lebanon-Jordan-Turkey slices) is the most-overlooked humanitarian plan globally in 2025** — Lebanon slice at $2.99B required, 9% funded. Invisible to traditional HRP↔HNO joins because the 3RP is UNHCR-coordinated; surfaced here via our FTS-as-backbone architecture.
- **Tier 1 (most overlooked) contains only 4 plans** — Lebanon-Syria-3RP, DR Congo, Yemen, Myanmar. All four sit at INFORM `operating_env_score` ≥ 8.5 (three at the 10.0 ceiling) — funding gap and access constraint are coupled, not independent.
- **Sudan ranks Tier 2 despite being the world's worst documented crisis** (INFORM severity 9.6, 30.4M PIN — both highest in our dataset) because at 40% funded it is *better resourced* than the four Tier 1 plans. The metric measures *mismatch*, not absolute severity.
- **Palestine has the largest absolute funding gap in 2026** ($3.58B raw, $2.77B annualized) AND the maximum INFORM `operating_env_score = 10.0`, while having zero rows in HNO 2024 / 2025 / 2026. The multi-output architecture surfaces this across three lenses (per-plan, country-rollup, operating environment) precisely because no single number captures it.
- **Mauritania (severity 6.1, trend Increasing, no formal plan)** appears in the secondary `documented_need_no_plan` table — invisible to any funding-gap analysis alone.
- **The bonus task is built, not sketched.** 4,927 monthly INFORM observations across 108 countries spanning Jan 2021 – Apr 2026. The scale-invariant `structural_neglect_v2` flag fires for 37 chronically-severe countries; 36 of 37 are already in our gap-score ranking (only DPRK is invisible), validating the FTS-as-backbone architecture. **Angola** is the standout rising-risk surfaced only by the temporal layer (10 of last 12 months Increasing, currently High, absent from top tiers).
- **INFORM rescaled from 1–5 to 1–10 in January 2026** (median jumped by 1.98×). Discovered while building the bonus task; we rebuilt every multi-year signal on categorical labels (scale-invariant). A data due-diligence finding.

## How to run this notebook

1. Attach Serverless compute
2. Click **Run All**
3. Wait ≈ 2.5 minutes — the six stages complete in sequence
4. View the output text logs at the paths printed at the end (durable workspace storage)
5. Open `gap_map.html` from the output directory for the interactive geospatial view
6. (Optional) Open `verification_test.py` from the companion file list and run as a final cell — should report 53/53 PASS

## Companion artifacts

- **WRITEUP.md** — full technical write-up (methodology, sensitivity, findings, limitations, biases, real-workflow integration)
- **FINDINGS.md** — executive Good/Bad/Ugly summary
- **GAP_SCORE_DESIGN.md** — design rationale with two rounds of assessor self-review
- **PDF_ALIGNMENT_ASSESSMENT.md** — section-by-section alignment with the challenge PDF
- **verification_test.py** — 53 automated pass/fail checks against every documented finding (run as final cell to audit)

## Pipeline stages

1. **Unified crisis table** — joins HRP + FTS + HNO with FTS plan-tied rows as the backbone (77 countries, 239 plan-rows)
2. **INFORM Severity Index — single snapshot** — adds standardized 0–10 severity, trend, reliability, operating-environment dimensions (68 countries, April 2026 file)
3. **Gap score + tier classification + three output tables** — `ranked_gap_scores` (239 rows), `country_rollup` (199 rows), `documented_need_no_plan` (8 rows); includes sensitivity test with measured Tier 1 set stability
4. **Bonus task — INFORM time series + temporal signals** — 4,927 monthly observations, 108 countries, scale-change detection, `structural_neglect_v2` and rising-risk signals merged into ranking
5. **Geospatial deliverable** — Plotly world map with 3 layers (Tier 1, Tier 2, documented-need-no-plan), saved as interactive HTML + static PNG
6. **Durable output saving** — persists Parquet/JSON/HTML files to workspace storage so artifacts survive cluster shutdown and an assessor can re-open them


In [0]:
%pip install openpyxl
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Stage 1 — Unified crisis table (FTS-as-backbone)

### What this stage does
Joins five OCHA datasets into a single country-year-plan table:
- **HRP** — Humanitarian Response Plans (plan codes, requirements, country/year coverage)
- **FTS country-level + cluster-level** — funding requirements + funding received per plan (cluster file used downstream for `hidden_sector_gap`)
- **HNO 2024 + 2025 + 2026** — People in Need (with HNO 2026 defensively skipped — file is structurally incomplete)
- **Population (admin0)** — country baselines

### Why FTS-as-backbone (not HRP)
An earlier version of this pipeline used HRP as the join backbone. The result captured only 22 countries — the ones with formal HRPs that have HNO documentation. **This missed the Syria 3RP entirely** (UNHCR-coordinated, not OCHA-HPC, no HNO footprint).

The current v2 architecture uses FTS plan-tied rows (`code IS NOT NULL`) as the backbone. This captures all plan types — HRP + Flash Appeal + Regional Response Plan — and surfaces 77 countries (3.5× the HRP-only baseline). The Syria 3RP, Gaza, Lebanon, and other refugee/flash-appeal crises all become visible.

### Output of this stage
- `crises` DataFrame — 239 rows × 14 columns spanning 2024–2026
- Each row = `(country, year, plan_code)` with `plan_type`, `fts_requirements`, `fts_funding`, `fts_pct_funded`, `country_unattributed_funding`, and HNO fields


In [0]:
# ============================================================
# Geo-Insight v2 loader
# Backbone: FTS plan-tied rows (HRP + Flash Appeal + Regional Plan)
# Layered: HNO most-recent-year per country (24/25/26 fallback)
# Tracked separately: "Not specified" loose funding flows per country-year
# ============================================================
import pandas as pd
import numpy as np

VOLUME = "/Volumes/cmu_hackathon/common/unocha"

# ============================================================
# 1) LOAD - all sources, skip HXL tag rows for HRP/HNO
# ============================================================
hrp = pd.read_csv(f"{VOLUME}/humanitarian-response-plans.csv", skiprows=[1], low_memory=False)
hno24 = pd.read_csv(f"{VOLUME}/hpc_hno_2024.csv", skiprows=[1], low_memory=False)
hno25 = pd.read_csv(f"{VOLUME}/hpc_hno_2025.csv", skiprows=[1], low_memory=False)
hno26 = pd.read_csv(f"{VOLUME}/hpc_hno_2026.csv", skiprows=[1], low_memory=False)
fts = pd.read_csv(f"{VOLUME}/fts_requirements_funding_global.csv", low_memory=False)

print(f"HRP:    {len(hrp):>7,} rows")
print(f"HNO 24: {len(hno24):>7,} rows")
print(f"HNO 25: {len(hno25):>7,} rows")
print(f"HNO 26: {len(hno26):>7,} rows")
print(f"FTS:    {len(fts):>7,} rows")

# ============================================================
# 2) HNO - build multi-year country rollups
# ============================================================
def hno_country_rollup(df, year):
    # Defensive: HNO 2026 has fewer columns (sector-only, no admin levels, no ALL rollup).
    # If the file doesn't have the rollup-eligible structure, skip it.
    required = {"Admin 1 PCode", "Cluster", "Category", "Country ISO3",
                "Population", "In Need", "Targeted", "Reached"}
    missing = required - set(df.columns)
    if missing or len(df) == 0:
        print(f"  HNO {year}: SKIPPED ({len(df)} rows, missing cols: {sorted(missing) or 'none'})")
        return pd.DataFrame(columns=["iso3", "hno_source_year", "population_hno",
                                     "people_in_need", "people_targeted", "people_reached"])
    rollup = df[
        df["Admin 1 PCode"].isna()
        & (df["Cluster"] == "ALL")
        & df["Category"].isna()
    ].copy()
    rollup = rollup.rename(columns={
        "Country ISO3": "iso3",
        "Population":   "population_hno",
        "In Need":      "people_in_need",
        "Targeted":     "people_targeted",
        "Reached":      "people_reached",
    })
    rollup["hno_source_year"] = year
    print(f"  HNO {year}: {len(rollup)} country rollup row(s)")
    return rollup[["iso3", "hno_source_year", "population_hno",
                   "people_in_need", "people_targeted", "people_reached"]]

hno_all = pd.concat([
    hno_country_rollup(hno24, 2024),
    hno_country_rollup(hno25, 2025),
    hno_country_rollup(hno26, 2026),
], ignore_index=True)

# Most recent year per country
hno_latest = (hno_all
    .sort_values("hno_source_year", ascending=False)
    .drop_duplicates(subset=["iso3"], keep="first")
    .reset_index(drop=True))

print(f"\nHNO concatenated: {len(hno_all)} rollup rows | unique countries: {hno_latest['iso3'].nunique()}")
print("HNO source-year distribution (after most-recent dedup):")
print(hno_latest["hno_source_year"].value_counts().sort_index().to_string())

# ============================================================
# 3) FTS - split plan-tied vs unattributed
# ============================================================
fts["year"] = fts["year"].astype("Int64")

# Plan-tied rows are the backbone
fts_plan = fts.dropna(subset=["code"]).copy()
fts_plan = fts_plan.rename(columns={
    "countryCode":   "iso3",
    "code":          "plan_code",
    "name":          "plan_name",
    "typeName":      "plan_type",
    "requirements":  "fts_requirements",
    "funding":       "fts_funding",
    "percentFunded": "fts_pct_funded",
})

# Fallback: derive plan_type from code prefix when typeName is null
mask_missing = fts_plan["plan_type"].isna() & fts_plan["plan_code"].notna()
prefix_map = {"H": "HRP", "F": "Flash appeal", "R": "Regional response plan"}
fts_plan.loc[mask_missing, "plan_type"] = (
    fts_plan.loc[mask_missing, "plan_code"].str[0].map(prefix_map).fillna("Other")
)

fts_plan = fts_plan[["iso3", "year", "plan_code", "plan_name", "plan_type",
                     "fts_requirements", "fts_funding", "fts_pct_funded"]]

# "Not specified" loose flows - aggregate per country-year
fts_unattributed = (fts[fts["code"].isna()]
    .groupby(["countryCode", "year"])["funding"].sum()
    .reset_index()
    .rename(columns={"countryCode": "iso3", "funding": "country_unattributed_funding"}))

print(f"\nFTS plan-tied rows: {len(fts_plan):,}")
print(f"FTS unattributed flows (country-year aggregated): {len(fts_unattributed):,}")

# ============================================================
# 4) JOIN - FTS plan-tied backbone + HNO latest + unattributed
# ============================================================
crises = fts_plan[fts_plan["year"].between(2024, 2026)].copy()

# Layer HNO need figures (most-recent year per country; same value across all of country's plans)
crises = crises.merge(hno_latest, on="iso3", how="left")

# Layer unattributed funding at country-year level
crises = crises.merge(fts_unattributed, on=["iso3", "year"], how="left")

print(f"\n{'='*70}")
print(f"v2 Crises table: {len(crises):,} rows")
print(f"Unique country-year combinations: {crises[['iso3','year']].drop_duplicates().shape[0]:,}")
print(f"Unique countries: {crises['iso3'].nunique()} (v1 backbone was 22 HNO countries)")
print(f"{'='*70}")

print("\nPlan type breakdown:")
print(crises["plan_type"].value_counts().to_string())

# ============================================================
# 5) VERIFICATION - Syria, Gaza, Lebanon, Ethiopia 2025
# ============================================================
print(f"\n{'='*70}")
print("Verification: SYR / PSE / LBN / ETH in v2 crises table (year 2025)")
print(f"{'='*70}")
for code in ["SYR", "PSE", "LBN", "ETH"]:
    sub = crises[(crises["iso3"] == code) & (crises["year"] == 2025)]
    print(f"\n--- {code} 2025 ({len(sub)} plan rows) ---")
    if len(sub) > 0:
        cols = ["plan_code", "plan_type", "plan_name",
                "fts_requirements", "fts_funding", "fts_pct_funded",
                "people_in_need", "hno_source_year",
                "country_unattributed_funding"]
        print(sub[cols].to_string(index=False))
    else:
        print("  (still no rows!)")

# ============================================================
# 6) TOP "OVERLOOKED" CANDIDATES - largest absolute funding gap
# ============================================================
print(f"\n{'='*70}")
print("Top 15 plans by absolute funding gap (requirements - funding), 2025")
print(f"{'='*70}")
top = crises[crises["year"] == 2025].copy()
top["funding_gap_usd"] = top["fts_requirements"].fillna(0) - top["fts_funding"].fillna(0)
top = top.sort_values("funding_gap_usd", ascending=False).head(15)
print(top[["iso3", "plan_code", "plan_type",
          "fts_requirements", "fts_funding", "fts_pct_funded",
          "funding_gap_usd", "people_in_need"]].to_string(index=False))


HRP:        910 rows
HNO 24: 387,819 rows
HNO 25: 318,259 rows
HNO 26:     133 rows
FTS:      3,836 rows
  HNO 2024: 24 country rollup row(s)
  HNO 2025: 22 country rollup row(s)
  HNO 2026: SKIPPED (133 rows, missing cols: ['Admin 1 PCode'])

HNO concatenated: 46 rollup rows | unique countries: 24
HNO source-year distribution (after most-recent dedup):
2024     2
2025    22

FTS plan-tied rows: 1,254
FTS unattributed flows (country-year aggregated): 2,577

v2 Crises table: 239 rows
Unique country-year combinations: 199
Unique countries: 77 (v1 backbone was 22 HNO countries)

Plan type breakdown:
Regional response plan                  134
Humanitarian needs and response plan     54
Flash appeal                             21
Other                                    17
Humanitarian response plan               13

Verification: SYR / PSE / LBN / ETH in v2 crises table (year 2025)

--- SYR 2025 (1 plan rows) ---
plan_code    plan_type                                                      

## Stage 2 — INFORM Severity Index

### What this stage does
Loads the INFORM Crisis Severity Index (ACAPS / EU-JRC) as a standardized 0–10 severity signal per country. INFORM is methodologically consistent across countries — unlike HNO's People-in-Need, which uses different baselines per country team.

### Why INFORM matters for the gap score
HNO PIN is the natural need signal but has three problems:
1. **Coverage gap** — only 22 countries with HNO 2025 country rollups
2. **Methodological inconsistency** — Sudan's 30M PIN and Afghanistan's 23M PIN aren't strictly comparable
3. **No data for Palestine or Lebanon** in any year

INFORM solves all three: 68 countries, standardized 0–10 scale, includes Gaza/Palestine and Lebanon. The severity score becomes our primary need-side signal; HNO PIN provides magnitude where available.

### Schema-drift handling
Across 7 years of INFORM data, the schema changed:
- 2021 file has a typo in the sheet name (`FORM Severity - country` missing "IN")
- 2021/2022 used `TYPE OF CRISIS`; 2023+ renamed to `DRIVERS`
- 2020 file has a different structure (skipped — outside our 2021+ window)

The loader handles all three defensively.

### Output of this stage
- `inform_country` DataFrame — 68 countries with: `severity_score` (0–10), `severity_trend` (Increasing/Stable/Decreasing), `reliability` (Very Low → Very High), and crisis drivers


In [0]:
# ============================================================
# INFORM Severity Index loader v1 — most recent file only
# Auto-selects the latest non-duplicate, non-mid-month INFORM file.
# Writes all output to /tmp/inform_loader_output.txt for easy sharing.
# ============================================================
import pandas as pd
import re
from pathlib import Path

VOLUME = "/Volumes/cmu_hackathon/common/unocha"
OUTPUT_FILE = "/tmp/inform_loader_output.txt"

# Output helper - both prints and buffers for the file
_buffer = []
def out(*args, sep=" "):
    msg = sep.join(str(a) for a in args)
    _buffer.append(msg)
    print(msg)

MONTH_NAMES = {
    "january":1,"february":2,"march":3,"april":4,"may":5,"june":6,
    "july":7,"august":8,"september":9,"october":10,"november":11,"december":12
}

def parse_data_year_month(name):
    """Extract year/month from the month-name in the filename (the actual data date)."""
    n = name.lower()
    for mname, mnum in MONTH_NAMES.items():
        m = re.search(rf"{mname}[-_ ](\d{{4}})", n)
        if m:
            return int(m.group(1)), mnum
    return None, None

# ============================================================
# 1) Auto-select the most recent usable INFORM file
# ============================================================
inform_files = []
for f in sorted(Path(VOLUME).glob("*.xlsx")):
    name = f.name.lower()
    if "inform" not in name and "severity" not in name:
        continue
    y, m = parse_data_year_month(f.name)
    if y is None:
        continue
    if y < 2021:        # 2020 files use an older format - skip
        continue
    if "(1)" in name:    # exact-duplicate suffix
        continue
    if "mid_" in name or "mid-" in name:
        continue
    inform_files.append({"path": f, "name": f.name, "year": y, "month": m})

inform_files.sort(key=lambda r: (r["year"], r["month"]))
latest = inform_files[-1]
out(f"Selected: {latest['name']}  (data date: {latest['year']}-{latest['month']:02d})")

# ============================================================
# 2) Read the country sheet (handle the 2021 typo sheet name too)
# ============================================================
xl = pd.ExcelFile(latest["path"])
candidates = [s for s in xl.sheet_names if s == "INFORM Severity - country"]
if not candidates:
    candidates = [s for s in xl.sheet_names if "FORM Severity - country" in s]
if not candidates:
    candidates = [s for s in xl.sheet_names
                  if "country" in s.lower() and "severity" in s.lower()]
sheet = candidates[0]
out(f"Sheet: {sheet!r}")

inform_raw = pd.read_excel(latest["path"], sheet_name=sheet, header=1, skiprows=[2, 3])
out(f"Raw shape: {inform_raw.shape}")

# ============================================================
# 3) Alias older column name (TYPE OF CRISIS -> DRIVERS)
# ============================================================
if "TYPE OF CRISIS" in inform_raw.columns and "DRIVERS" not in inform_raw.columns:
    inform_raw = inform_raw.rename(columns={"TYPE OF CRISIS": "DRIVERS"})

# ============================================================
# 4) Clean / select columns
# ============================================================
inform = inform_raw[inform_raw["ISO3"].notna()].copy()

keep_cols = {
    "ISO3":                          "iso3",
    "COUNTRY":                       "country_name",
    "CRISIS":                        "crisis_name",
    "DRIVERS":                       "crisis_drivers",
    "INFORM Severity Index":         "severity_score",
    "INFORM Severity category":      "severity_category_num",
    "INFORM Severity category.1":    "severity_category_lbl",
    "Trend (last 3 months)":         "severity_trend",
    "Reliability":                   "reliability",
    "Impact of the crisis":          "impact_score",
    "Conditions of people affected": "conditions_score",
    "Complexity of the crisis":      "complexity_score",
    "People in need":                "pin_score",
    "Last updated":                  "last_updated",
}
available = {k: v for k, v in keep_cols.items() if k in inform.columns}
inform_country = inform[list(available.keys())].rename(columns=available).copy()
inform_country["inform_data_year"] = latest["year"]
inform_country["inform_data_month"] = latest["month"]

for c in ["severity_score", "severity_category_num", "impact_score",
          "conditions_score", "complexity_score", "pin_score"]:
    if c in inform_country.columns:
        inform_country[c] = pd.to_numeric(inform_country[c], errors="coerce")

out(f"\nCleaned inform_country: {inform_country.shape}")
out(f"Columns: {list(inform_country.columns)}")

# ============================================================
# 5) Sanity checks
# ============================================================
out(f"\n--- Severity score distribution ---")
out(inform_country["severity_score"].describe().to_string())

out(f"\n--- Top 15 by INFORM severity ---")
cols = ["iso3", "country_name", "crisis_name", "crisis_drivers",
        "severity_score", "severity_category_lbl", "severity_trend", "reliability"]
top15 = inform_country.sort_values("severity_score", ascending=False).head(15)
out(top15[[c for c in cols if c in top15.columns]].to_string(index=False))

# ============================================================
# 6) Verification: SYR / PSE / LBN / ETH
# ============================================================
out(f"\n--- Verification: SYR / PSE / LBN / ETH ---")
for code in ["SYR", "PSE", "LBN", "ETH"]:
    rows = inform_country[inform_country["iso3"] == code]
    out(f"\n{code} ({len(rows)} row(s)):")
    if len(rows) > 0:
        cs = ["country_name", "crisis_name", "crisis_drivers", "severity_score",
              "severity_category_lbl", "severity_trend", "reliability", "pin_score"]
        out(rows[[c for c in cs if c in rows.columns]].to_string(index=False))

# ============================================================
# 7) Overlap with our crises table
# ============================================================
try:
    fts_countries = set(crises["iso3"].dropna().unique())
    inform_countries = set(inform_country["iso3"].dropna().unique())
    overlap = fts_countries & inform_countries
    out(f"\n--- Overlap with crises table ---")
    out(f"  Countries in crises (FTS-driven): {len(fts_countries)}")
    out(f"  Countries in INFORM:              {len(inform_countries)}")
    out(f"  Overlap:                          {len(overlap)}")
    out(f"  In crises but NOT in INFORM:      {sorted(fts_countries - inform_countries)}")
    out(f"  In INFORM but NOT in crises:      {sorted(inform_countries - fts_countries)}")
except NameError:
    out("\n(crises table not in memory - skipping overlap check)")

# ============================================================
# 8) Save buffer to file
# ============================================================
with open(OUTPUT_FILE, "w") as f:
    f.write("\n".join(_buffer))

print(f"\n\n>>> Output written to {OUTPUT_FILE}")
print(f">>> Total size: {len(open(OUTPUT_FILE).read()):,} chars")
print(f">>> To view all content as plain text, run in a new cell:")
print(f">>>   print(open('{OUTPUT_FILE}').read())")


Selected: 202604-inform-severity-april-2026.xlsx  (data date: 2026-04)
Sheet: 'INFORM Severity - country'
Raw shape: (68, 21)

Cleaned inform_country: (68, 16)
Columns: ['iso3', 'country_name', 'crisis_name', 'crisis_drivers', 'severity_score', 'severity_category_num', 'severity_category_lbl', 'severity_trend', 'reliability', 'impact_score', 'conditions_score', 'complexity_score', 'pin_score', 'last_updated', 'inform_data_year', 'inform_data_month']

--- Severity score distribution ---
count    68.000000
mean      6.495588
std       1.787138
min       3.000000
25%       5.275000
50%       6.600000
75%       8.000000
max       9.600000

--- Top 15 by INFORM severity ---
iso3 country_name                                  crisis_name                                                                                          crisis_drivers  severity_score severity_category_lbl severity_trend reliability
 SDN        Sudan                      Complex crisis in Sudan                            

## Stage 3 — Gap score, tier classification, three output tables

### The formula

```
gap_score = severity_norm × need_norm × coverage_gap_norm
```

A multiplicative score in [0, 1] per `(country, year, plan_code)` row. Multiplicative because all three components are *necessary* for "overlooked" — a crisis with zero severity, zero need, or full funding is not overlooked.

### Components

| Component | Source | Notes |
|---|---|---|
| `severity_norm` | INFORM Severity Index / 10 | 0–1, primary need-side signal |
| `need_norm_absolute` | log10(PIN) / 8 for HRPs, log10(requirements_USD / 1M) / 4 for Flash Appeals + Regional Plans | Plan-type-aware to avoid PIN inheritance on narrow-scope plans |
| `coverage_gap_norm` | 1 − fts_pct_funded / 100 (annualized for in-progress 2026 with year_progress ≥ 0.20 guard) | Raw and annualized exposed as separate columns |

### Tier classification

To avoid false precision (the PDF warns against it), scores are bucketed:

| Tier | Range | Meaning |
|---|---|---|
| Tier 1 | gap_score ≥ 0.55 | Most overlooked |
| Tier 2 | 0.45 – 0.55 | Severely overlooked |
| Tier 3 | 0.30 – 0.45 | Underfunded |
| Tier 4 | < 0.30 | Adequately resourced |

### Per-row honesty (uncertainty surfaced, not blended in)

- `confidence_band` (high / medium / low) — derived from `reliability` + need source + year status
- `severity_source`, `need_source`, `hno_source_year`, `year_status` — provenance flags
- `operating_env_score`, `society_safety_score` — parallel dimensions (NOT mixed into gap_score; displayed alongside)

Confidence is **never multiplied into the score**. Multiplying would artificially de-rank cases we should be most alarmed about (we don't know enough about them).

### Three output tables produced

| Table | Rows | What it answers |
|---|---|---|
| `ranked_gap_scores` | 239 | Which plans are most overlooked? (sorted by gap_score) |
| `country_rollup` | 199 | Which countries have the largest total funding gap? (sorted by absolute gap) |
| `documented_need_no_plan` | 8 | Which countries have documented need but no formal response plan? |

### Why three tables, not one

The PDF asks for "a ranked list of crises *or countries*." Country-level aggregation hides multi-plan dynamics (Lebanon has FLBN25 at 78% AND RSYR25 at 9% — a single country-level rank obscures both). Plan-level ranking misses countries with no plan (Mauritania). Three tables answer three distinct questions.

### Sensitivity disclosure with measured Tier 1 set stability

The pipeline runs a ±0.05 perturbation test on each component independently and combined. Both the top-10 ordinal stability AND the Tier 1 set stability are measured (5 trials per scenario, 4 baseline Tier 1 plans):

| Scenario | Top-10 mean preserved | Tier 1 set preservation | Reading |
|---|---|---|---|
| severity_only | 8.4 / 10 | **[4,4,4,4,4] / 4 — fully preserved** | Tier 1 stable under severity noise |
| need_only | 7.8 / 10 | [4,3,4,4,3] / 4 — 3.6 mean | Partial |
| coverage_only | 7.8 / 10 | [4,4,4,3,4] / 4 — 3.8 mean | Partial |
| all_three combined | 6.6 / 10 | [3,3,4,3,3] / 4 — 3.2 mean | Partial (3 of 4 always preserved) |

The threshold-edge plan that drops in the partial-preservation trials is **MMR-HMMR25b at gap_score 0.567** — closest to the 0.55 Tier 1 cutoff. The tier-set reading is the right output unit; exact intra-tier rank order should not be over-interpreted.


In [0]:
# ============================================================
# GAP SCORE v1 — per GAP_SCORE_DESIGN.md spec
#
# Inputs (expected in memory):
#   - crises          (from loader_cell_v2.py)
#   - inform_country  (from inform_loader_v1.py)
#
# Outputs (in memory + on disk):
#   - ranked_gap_scores              (DataFrame + /tmp/gap_score_v1_ranked.parquet)
#   - country_rollup                 (DataFrame + /tmp/gap_score_v1_country_rollup.parquet)
#   - documented_need_no_plan        (DataFrame + /tmp/gap_score_v1_no_plan.parquet)
#   - sensitivity_report             (dict + /tmp/gap_score_v1_sensitivity.json)
#   - human-readable log             (/tmp/gap_score_v1_output.txt)
# ============================================================
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime
from pathlib import Path

VOLUME = "/Volumes/cmu_hackathon/common/unocha"
OUTPUT_TXT     = "/tmp/gap_score_v1_output.txt"
OUTPUT_RANKED  = "/tmp/gap_score_v1_ranked.parquet"
OUTPUT_ROLLUP  = "/tmp/gap_score_v1_country_rollup.parquet"
OUTPUT_NOPLAN  = "/tmp/gap_score_v1_no_plan.parquet"
OUTPUT_SENS    = "/tmp/gap_score_v1_sensitivity.json"

ANALYSIS_DATE = datetime(2026, 5, 18)

_buf = []
def out(*a, sep=" "):
    msg = sep.join(str(x) for x in a)
    _buf.append(msg)
    print(msg)

out("=" * 70)
out("GAP SCORE v1")
out("=" * 70)

# ============================================================
# 1) Country centroids (UN M49 source, embedded)
# ============================================================
CENTROIDS = {
    "AFG": (33.939, 67.710),    "AGO": (-11.203, 17.874),   "ARG": (-38.416, -63.617),
    "BDI": (-3.373, 29.919),    "BEN": (9.308, 2.316),      "BFA": (12.238, -1.562),
    "BGD": (23.685, 90.356),    "BGR": (42.734, 25.486),    "BOL": (-16.290, -63.589),
    "BRA": (-14.235, -51.925),  "CAF": (6.611, 20.939),     "CHL": (-35.675, -71.543),
    "CMR": (7.370, 12.355),     "COD": (-4.038, 21.759),    "COG": (-0.228, 15.828),
    "COL": (4.571, -74.297),    "CRI": (9.749, -83.753),    "CUW": (12.170, -68.990),
    "CZE": (49.817, 15.473),    "DJI": (11.825, 42.590),    "DOM": (18.736, -70.163),
    "ECU": (-1.831, -78.183),   "EGY": (26.821, 30.802),    "EST": (58.595, 25.014),
    "ETH": (9.145, 40.490),     "GTM": (15.783, -90.231),   "GUY": (4.860, -58.930),
    "HND": (15.200, -86.242),   "HTI": (18.971, -72.285),   "HUN": (47.162, 19.503),
    "IRN": (32.428, 53.688),    "JOR": (30.585, 36.238),    "KEN": (-0.024, 37.906),
    "LBN": (33.855, 35.862),    "LBY": (26.335, 17.228),    "LTU": (55.169, 23.881),
    "LVA": (56.880, 24.603),    "MDA": (47.412, 28.370),    "MEX": (23.635, -102.553),
    "MLI": (17.571, -3.996),    "MMR": (21.914, 95.956),    "MOZ": (-18.666, 35.530),
    "MWI": (-13.254, 34.302),   "NER": (17.608, 8.082),     "NGA": (9.082, 8.675),
    "PAK": (30.375, 69.345),    "PAN": (8.538, -80.782),    "PER": (-9.190, -75.015),
    "PHL": (12.880, 121.774),   "POL": (51.919, 19.145),    "PRY": (-23.443, -58.444),
    "PSE": (31.952, 35.233),    "ROU": (45.943, 24.967),    "RWA": (-1.940, 29.874),
    "SDN": (12.863, 30.218),    "SLV": (13.794, -88.897),   "SOM": (5.152, 46.200),
    "SSD": (6.877, 31.307),     "SVK": (48.669, 19.699),    "SYR": (34.802, 38.997),
    "TCD": (15.454, 18.732),    "TTO": (10.692, -61.223),   "TUR": (38.964, 35.243),
    "TZA": (-6.369, 34.889),    "UGA": (1.373, 32.290),     "UKR": (48.379, 31.166),
    "URY": (-32.523, -55.766),  "VEN": (6.424, -66.590),    "VNM": (14.058, 108.277),
    "YEM": (15.553, 48.516),    "ZMB": (-13.134, 27.849),   "ZWE": (-19.015, 29.155),
    "ABW": (12.521, -69.968),
}

# ============================================================
# 2) Validate required inputs are in memory
# ============================================================
try:
    _ = crises
except NameError:
    raise RuntimeError(
        "Required variable 'crises' not found. "
        "Run loader_cell_v2.py first, then re-run this cell."
    )
try:
    _ = inform_country
except NameError:
    raise RuntimeError(
        "Required variable 'inform_country' not found. "
        "Run inform_loader_v1.py first, then re-run this cell."
    )
out(f"\nInputs OK: crises={len(crises)} rows, inform_country={len(inform_country)} rows")

# ============================================================
# 3) Load FTS cluster (sector) data for hidden_sector_gap
# ============================================================
fts_cluster = pd.read_csv(
    f"{VOLUME}/fts_requirements_funding_globalcluster_global.csv", low_memory=False
)
fts_cluster = fts_cluster.dropna(subset=["code"]).copy()
fts_cluster = fts_cluster.rename(columns={
    "countryCode":   "iso3",
    "code":          "plan_code",
    "cluster":       "sector",
    "requirements":  "sector_requirements",
    "funding":       "sector_funding",
    "percentFunded": "sector_pct_funded",
})
fts_cluster["year"] = fts_cluster["year"].astype("Int64")
out(f"FTS cluster: {len(fts_cluster)} rows")

# ============================================================
# 4) Re-load latest INFORM file to fetch operating_env + society_safety
#    (inform_country from prior cell only kept a subset of columns)
# ============================================================
MONTH_NAMES = {"january":1,"february":2,"march":3,"april":4,"may":5,"june":6,
               "july":7,"august":8,"september":9,"october":10,"november":11,"december":12}

def parse_data_ym(name):
    n = name.lower()
    for mn, mv in MONTH_NAMES.items():
        m = re.search(rf"{mn}[-_ ](\d{{4}})", n)
        if m:
            return int(m.group(1)), mv
    return None, None

inform_xlsx = []
for f in sorted(Path(VOLUME).glob("*.xlsx")):
    n = f.name.lower()
    if "inform" not in n and "severity" not in n:
        continue
    y, mo = parse_data_ym(f.name)
    if y is None or y < 2021 or "(1)" in n or "mid_" in n or "mid-" in n:
        continue
    inform_xlsx.append({"path": f, "year": y, "month": mo})
inform_xlsx.sort(key=lambda r: (r["year"], r["month"]))
latest_inform = inform_xlsx[-1]
out(f"INFORM source: {latest_inform['path'].name}")

xl = pd.ExcelFile(latest_inform["path"])
candidates = [s for s in xl.sheet_names if s == "INFORM Severity - country"]
if not candidates:
    candidates = [s for s in xl.sheet_names if "FORM Severity - country" in s]
sheet = candidates[0]
inform_full = pd.read_excel(latest_inform["path"], sheet_name=sheet, header=1, skiprows=[2, 3])
if "TYPE OF CRISIS" in inform_full.columns and "DRIVERS" not in inform_full.columns:
    inform_full = inform_full.rename(columns={"TYPE OF CRISIS": "DRIVERS"})
inform_full = inform_full[inform_full["ISO3"].notna()].copy()
inform_extra = inform_full[["ISO3", "Operating environment", "Society and safety"]].rename(columns={
    "ISO3": "iso3",
    "Operating environment": "operating_env_score",
    "Society and safety":    "society_safety_score",
})
for c in ["operating_env_score", "society_safety_score"]:
    inform_extra[c] = pd.to_numeric(inform_extra[c], errors="coerce")
out(f"INFORM extra dims: {len(inform_extra)} rows")

# ============================================================
# 5) Build enriched table — Stage A from §8.3
# ============================================================
inform_join = inform_country[[
    "iso3", "country_name", "crisis_name", "crisis_drivers",
    "severity_score", "severity_category_lbl",
    "severity_trend", "reliability", "pin_score",
]].rename(columns={
    "severity_score":         "inform_severity",
    "severity_category_lbl":  "inform_severity_category",
    "severity_trend":         "inform_trend",
    "reliability":            "inform_reliability",
    "pin_score":              "inform_pin_score",
})

enriched = (crises
    .merge(inform_join, on="iso3", how="left")
    .merge(inform_extra, on="iso3", how="left"))

# country_name fallback if not in INFORM
enriched["country_name"] = enriched["country_name"].fillna(enriched["iso3"])

# centroids
enriched["latitude"]  = enriched["iso3"].map(lambda x: CENTROIDS.get(x, (None, None))[0])
enriched["longitude"] = enriched["iso3"].map(lambda x: CENTROIDS.get(x, (None, None))[1])

# hidden_sector_gap from fts_cluster
def worst_sector(g):
    if g["sector_pct_funded"].notna().sum() == 0:
        return pd.Series({"worst_sector_name": None, "worst_sector_pct_funded": None})
    idx = g["sector_pct_funded"].idxmin()
    return pd.Series({
        "worst_sector_name":       g.loc[idx, "sector"],
        "worst_sector_pct_funded": g.loc[idx, "sector_pct_funded"],
    })

sector_summary = (fts_cluster
    .groupby(["iso3", "plan_code", "year"], as_index=False)
    .apply(worst_sector)
    .reset_index(drop=True))

enriched = enriched.merge(sector_summary, on=["iso3", "plan_code", "year"], how="left")
enriched["hidden_sector_gap"] = (
    (enriched["worst_sector_pct_funded"].fillna(100) < 15)
    & (enriched["fts_pct_funded"].fillna(0) > 30)
)

# ============================================================
# 6) Compute components — Stage B from §8.3
# ============================================================
# severity_norm
def sev(row):
    if pd.notna(row["inform_severity"]):
        return row["inform_severity"] / 10, "INFORM"
    if pd.notna(row["inform_pin_score"]):
        return row["inform_pin_score"] / 10, "inform_pin_fallback"
    return 0.5, "default"
sev_data = enriched.apply(sev, axis=1)
enriched["severity_norm"]   = sev_data.map(lambda x: x[0])
enriched["severity_source"] = sev_data.map(lambda x: x[1])

# need_norm_absolute — PLAN-TYPE AWARE
# HRPs cover the full country response -> use HNO PIN when available
# Regional / Flash / Other plans cover a narrower sub-population -> use FTS requirements
# (don't inherit the country's total PIN onto narrow-scope plans like Syria 3RP slices
#  or Ethiopia regional refugee plans, which would over-inflate need)
HRP_TYPES = {"Humanitarian needs and response plan", "Humanitarian response plan"}

def need(row):
    is_full_country_plan = row.get("plan_type") in HRP_TYPES
    pin_ok = pd.notna(row["people_in_need"]) and row["people_in_need"] > 0
    req_ok = pd.notna(row["fts_requirements"]) and row["fts_requirements"] > 0
    if is_full_country_plan and pin_ok:
        return min(np.log10(max(row["people_in_need"], 1)) / 8, 1.0), "HNO_PIN"
    if req_ok:
        return min(np.log10(max(row["fts_requirements"] / 1e6, 1)) / 4, 1.0), "FTS_requirements_proxy"
    # Last resort: HRP with no PIN and no requirements -> 0
    if pin_ok:
        return min(np.log10(max(row["people_in_need"], 1)) / 8, 1.0), "HNO_PIN"
    return 0.0, "none"

need_data = enriched.apply(need, axis=1)
enriched["need_norm_absolute"] = need_data.map(lambda x: x[0])
enriched["need_source"]        = need_data.map(lambda x: x[1])

# need_norm_relative = PIN / population
enriched["need_norm_relative"] = np.where(
    (enriched["people_in_need"].notna()) & (enriched["population_hno"].fillna(0) > 0),
    enriched["people_in_need"] / enriched["population_hno"],
    np.nan,
)

# coverage_gap (raw + annualized) with year_progress guard
def days_elapsed(year):
    if year < ANALYSIS_DATE.year: return 365
    if year > ANALYSIS_DATE.year: return 0
    return (ANALYSIS_DATE - datetime(year, 1, 1)).days

def cov_gap(row):
    pct = row["fts_pct_funded"]
    yr  = int(row["year"])
    status = "closed" if yr < ANALYSIS_DATE.year else "in_progress" if yr == ANALYSIS_DATE.year else "future"
    if pd.isna(pct):
        return None, None, status
    pct = max(0, min(pct, 100))
    raw = 1 - pct / 100
    yp = days_elapsed(yr) / 365
    if status == "in_progress" and yp >= 0.20:
        expected = (pct / 100) / yp
        annualized = 1 - max(0, min(expected, 1.0))
    else:
        annualized = raw
    return raw, annualized, status

cov_data = enriched.apply(cov_gap, axis=1)
enriched["coverage_gap_raw"]        = cov_data.map(lambda x: x[0])
enriched["coverage_gap_annualized"] = cov_data.map(lambda x: x[1])
enriched["year_status"]             = cov_data.map(lambda x: x[2])

# ============================================================
# 7) Plan family + structural neglect — Stage C from §8.3
# ============================================================
def plan_family(code):
    if pd.isna(code):
        return None
    m = re.match(r"^(.+?)\d{2}[a-z]?$", str(code), re.IGNORECASE)
    return m.group(1) if m else str(code)

enriched["plan_family"] = enriched["plan_code"].map(plan_family)

# Coverage history lookup: (iso3, plan_family, year) -> max fts_pct_funded
hist = (enriched
    .dropna(subset=["plan_family", "fts_pct_funded"])
    .groupby(["iso3", "plan_family", "year"])["fts_pct_funded"]
    .max().to_dict())

def structural(row):
    if pd.isna(row["plan_family"]) or pd.isna(row["inform_severity"]):
        return False
    if row["inform_severity"] < 7:
        return False
    iso, fam, yr = row["iso3"], row["plan_family"], int(row["year"])
    priors = [hist.get((iso, fam, yr - 1)), hist.get((iso, fam, yr - 2))]
    valid = [c for c in priors if c is not None]
    if len(valid) < 2:
        return False
    return all(c < 50 for c in valid)

enriched["structural_neglect_flag"] = enriched.apply(structural, axis=1)

# coverage_inclusive_of_unattributed
def cov_incl(row):
    req = row["fts_requirements"]
    if pd.isna(req) or req == 0:
        return None
    fund = row["fts_funding"] if pd.notna(row["fts_funding"]) else 0
    unat = row["country_unattributed_funding"] if pd.notna(row["country_unattributed_funding"]) else 0
    return (fund + 0.5 * unat) / req

enriched["coverage_inclusive_of_unattributed"] = enriched.apply(cov_incl, axis=1)

# ============================================================
# 8) Gap score + confidence + explanation — Stage D from §8.3
# ============================================================
enriched["gap_score"] = (
    enriched["severity_norm"].fillna(0)
    * enriched["need_norm_absolute"].fillna(0)
    * enriched["coverage_gap_annualized"].fillna(0)
)

# Tier classification — avoid false precision by grouping scores into bands.
# Scores within ~0.10 of each other are statistically indistinguishable
# given measurement noise; tiers prevent over-interpretation of exact rank.
def gap_tier(score):
    if pd.isna(score):
        return "Tier 4 (insufficient data)"
    if score >= 0.55:
        return "Tier 1 (most overlooked)"
    if score >= 0.45:
        return "Tier 2 (severely overlooked)"
    if score >= 0.30:
        return "Tier 3 (underfunded)"
    return "Tier 4 (adequately resourced)"

enriched["gap_tier"] = enriched["gap_score"].apply(gap_tier)

def confidence_band(row):
    s = 0
    rel = row.get("inform_reliability")
    if rel in ("Very High", "High"): s += 2
    elif rel == "Medium":            s += 1
    if row.get("need_source") == "HNO_PIN":      s += 1
    if row.get("year_status") == "closed":       s += 1
    return "high" if s >= 3 else "medium" if s == 2 else "low"

enriched["confidence_band"] = enriched.apply(confidence_band, axis=1)

# Explanation
def explain(row):
    flags = []
    if row.get("structural_neglect_flag"):
        flags.append("Structural neglect: 2+ years <50% coverage.")
    if row.get("hidden_sector_gap"):
        flags.append(f"Hidden sector gap: {row['worst_sector_name']} at {row['worst_sector_pct_funded']:.0f}%.")
    op_env = row.get("operating_env_score")
    if pd.notna(op_env) and op_env >= 7.5:
        flags.append(f"Constrained operating environment: {op_env:.1f}/10 — funding alone may not enable delivery.")
    flags_str = " ".join(flags)
    pin_or_req = (
        f"{row['people_in_need']:,.0f} PIN"
        if pd.notna(row["people_in_need"])
        else f"${row['fts_requirements']/1e6:,.0f}M required"
        if pd.notna(row["fts_requirements"])
        else "need unknown"
    )
    sev_str = f"{row['inform_severity']:.1f}/10" if pd.notna(row.get("inform_severity")) else "n/a"
    trend = row.get("inform_trend", "?")
    cat = row.get("inform_severity_category", "?")
    pct = f"{row['fts_pct_funded']:.0f}%" if pd.notna(row.get("fts_pct_funded")) else "n/a"
    return (
        f"{row['country_name']} {row['plan_code']} ({row['plan_type']}) — "
        f"severity {sev_str} ({cat}, trend {trend}), "
        f"need from {row['need_source']} ({pin_or_req}), "
        f"coverage {pct} ({row['year_status']}). "
        f"Gap score {row['gap_score']:.2f} ({row['gap_tier']}). "
        f"Confidence: {row['confidence_band']}. {flags_str}"
    ).strip()

enriched["explanation"] = enriched.apply(explain, axis=1)

# ============================================================
# 9) Table 1: ranked_gap_scores — Stage E
# ============================================================
output_cols = [
    "iso3", "country_name", "plan_code", "plan_type", "year",
    "gap_score", "gap_tier",
    "severity_norm", "need_norm_absolute", "coverage_gap_annualized",
    "need_norm_relative", "coverage_gap_raw",
    "severity_source", "need_source", "hno_source_year", "year_status",
    "inform_reliability", "confidence_band",
    "inform_trend",
    "operating_env_score", "society_safety_score",
    "hidden_sector_gap", "worst_sector_name", "worst_sector_pct_funded",
    "structural_neglect_flag", "plan_family",
    "fts_requirements", "fts_funding", "fts_pct_funded",
    "coverage_inclusive_of_unattributed",
    "people_in_need", "population_hno",
    "country_unattributed_funding",
    "latitude", "longitude",
    "explanation",
]
ranked_gap_scores = (enriched[output_cols]
    .sort_values("gap_score", ascending=False)
    .reset_index(drop=True))
out(f"\nranked_gap_scores: {len(ranked_gap_scores)} rows")

# ============================================================
# 10) Table 2: country_rollup
# ============================================================
CB_ORDER = {"low": 0, "medium": 1, "high": 2}
CB_INV   = {v: k for k, v in CB_ORDER.items()}

def country_agg(g):
    sum_req  = g["fts_requirements"].fillna(0).sum()
    sum_fund = g["fts_funding"].fillna(0).sum()
    underf   = (g["fts_pct_funded"].fillna(100) < 50).sum()
    min_cb   = min((CB_ORDER.get(c, 1) for c in g["confidence_band"]), default=1)
    worst_idx = g["fts_pct_funded"].idxmin() if g["fts_pct_funded"].notna().any() else g.index[0]
    return pd.Series({
        "country_name":         g["country_name"].iloc[0],
        "severity_score":       g["inform_severity"].iloc[0] if pd.notna(g["inform_severity"].iloc[0]) else None,
        "severity_trend":       g["inform_trend"].iloc[0],
        "n_plans":              len(g),
        "n_plans_underfunded":  int(underf),
        "sum_fts_requirements": float(sum_req),
        "sum_fts_funding":      float(sum_fund),
        "sum_funding_gap_usd":  float(sum_req - sum_fund),
        "country_coverage_pct": (sum_fund / sum_req * 100) if sum_req > 0 else None,
        "worst_plan_coverage_pct": g["fts_pct_funded"].min(),
        "worst_plan_code":      g.loc[worst_idx, "plan_code"],
        "max_gap_score":        g["gap_score"].max(),
        "mean_gap_score":       g["gap_score"].mean(),
        "gap_tier":             gap_tier(g["gap_score"].max()),
        "operating_env_score":  g["operating_env_score"].iloc[0] if pd.notna(g["operating_env_score"].iloc[0]) else None,
        "society_safety_score": g["society_safety_score"].iloc[0] if pd.notna(g["society_safety_score"].iloc[0]) else None,
        "any_structural_neglect": bool(g["structural_neglect_flag"].any()),
        "any_hidden_sector_gap":  bool(g["hidden_sector_gap"].any()),
        "confidence_band":      CB_INV[min_cb],
        "latitude":             g["latitude"].iloc[0],
        "longitude":            g["longitude"].iloc[0],
    })

country_rollup = (enriched
    .groupby(["iso3", "year"], as_index=False)
    .apply(country_agg)
    .reset_index(drop=True)
    .sort_values("sum_funding_gap_usd", ascending=False)
    .reset_index(drop=True))
out(f"country_rollup: {len(country_rollup)} rows")

# ============================================================
# 11) Table 3: documented_need_no_plan
# ============================================================
countries_with_plan = set(
    enriched[
        enriched["plan_code"].notna()
        & (enriched["fts_requirements"].fillna(0) > 0)
        & enriched["year"].between(2024, 2026)
    ]["iso3"].unique()
)

documented_need_no_plan = inform_country[
    (inform_country["severity_score"] >= 5)
    & (~inform_country["iso3"].isin(countries_with_plan))
].copy()
documented_need_no_plan["latitude"]  = documented_need_no_plan["iso3"].map(lambda x: CENTROIDS.get(x, (None, None))[0])
documented_need_no_plan["longitude"] = documented_need_no_plan["iso3"].map(lambda x: CENTROIDS.get(x, (None, None))[1])
documented_need_no_plan = (documented_need_no_plan
    .sort_values("severity_score", ascending=False)
    .reset_index(drop=True))
out(f"documented_need_no_plan: {len(documented_need_no_plan)} rows")

# ============================================================
# 12) Sensitivity analysis — Stage F
# ============================================================
np.random.seed(42)
baseline_top10 = set(
    ranked_gap_scores.head(10)
    .apply(lambda r: f"{r['iso3']}|{r['plan_code']}|{r['year']}", axis=1)
)
# Baseline Tier 1 set — the 4 plans the headline rests on
baseline_tier1 = set(
    ranked_gap_scores[ranked_gap_scores["gap_tier"] == "Tier 1 (most overlooked)"]
    .apply(lambda r: f"{r['iso3']}|{r['plan_code']}|{r['year']}", axis=1)
)

def perturb(scenario, cols, n_iter=5):
    top10_rates = []
    tier1_membership_rates = []  # how many of the 4 baseline Tier 1 plans remain >= 0.55 after perturb
    for _ in range(n_iter):
        p = enriched.copy()
        for c in cols:
            noise = np.random.uniform(-0.05, 0.05, size=len(p))
            p[c] = (p[c].fillna(0) + noise).clip(0, 1)
        p["gap_perturbed"] = (
            p["severity_norm"].fillna(0)
            * p["need_norm_absolute"].fillna(0)
            * p["coverage_gap_annualized"].fillna(0)
        )
        # Top-10 overlap (original measurement)
        new_top10 = set(
            p.nlargest(10, "gap_perturbed")
            .apply(lambda r: f"{r['iso3']}|{r['plan_code']}|{r['year']}", axis=1)
        )
        top10_rates.append(len(baseline_top10 & new_top10))
        # Tier 1 set stability: how many baseline Tier 1 plans still score >= 0.55
        perturbed_tier1 = set(
            p[p["gap_perturbed"] >= 0.55]
            .apply(lambda r: f"{r['iso3']}|{r['plan_code']}|{r['year']}", axis=1)
        )
        tier1_membership_rates.append(len(baseline_tier1 & perturbed_tier1))
    return {
        "scenario": scenario,
        "preservation_rates": top10_rates,
        "mean_preserved_in_top_10": float(np.mean(top10_rates)),
        "passes_8_of_10_threshold": all(r >= 8 for r in top10_rates),
        "tier1_set_membership_rates": tier1_membership_rates,
        "mean_tier1_baseline_plans_still_tier1": float(np.mean(tier1_membership_rates)),
        "tier1_set_fully_preserved": all(r == len(baseline_tier1) for r in tier1_membership_rates),
    }

sensitivity_report = {
    "method": "absolute +/- 0.05 uniform noise on [0,1] components",
    "n_iter_per_scenario": 5,
    "scenarios": [
        perturb("severity_only", ["severity_norm"]),
        perturb("need_only",     ["need_norm_absolute"]),
        perturb("coverage_only", ["coverage_gap_annualized"]),
        perturb("all_three",     ["severity_norm", "need_norm_absolute", "coverage_gap_annualized"]),
    ],
}

out(f"\nSensitivity results (top-10 stability + Tier 1 set stability):")
n_tier1 = len(baseline_tier1)
out(f"  Baseline Tier 1 set ({n_tier1} plans): {sorted(baseline_tier1)}")
for s in sensitivity_report["scenarios"]:
    out(f"  {s['scenario']:18s}  "
        f"top10 mean {s['mean_preserved_in_top_10']:.1f}/10 ({'PASS' if s['passes_8_of_10_threshold'] else 'FAIL'})  "
        f"|  Tier1 set: {s['tier1_set_membership_rates']} / {n_tier1} "
        f"({'fully preserved' if s['tier1_set_fully_preserved'] else 'partial'})")

# ============================================================
# 13) Print headline outputs
# ============================================================
out(f"\n{'='*70}\nTop 15 ranked_gap_scores\n{'='*70}")
top_cols = ["iso3", "plan_code", "plan_type", "year", "gap_score", "gap_tier",
            "severity_norm", "need_norm_absolute", "coverage_gap_annualized",
            "operating_env_score", "confidence_band",
            "structural_neglect_flag", "hidden_sector_gap"]
out(ranked_gap_scores.head(15)[top_cols].to_string(index=False))

out(f"\n{'='*70}\nTier distribution\n{'='*70}")
out(ranked_gap_scores["gap_tier"].value_counts().sort_index().to_string())

out(f"\n{'='*70}\nTop 15 country_rollup (by sum_funding_gap_usd)\n{'='*70}")
roll_cols = ["iso3", "country_name", "year", "n_plans", "n_plans_underfunded",
             "sum_funding_gap_usd", "country_coverage_pct", "max_gap_score",
             "gap_tier", "operating_env_score", "society_safety_score",
             "any_structural_neglect", "confidence_band"]
out(country_rollup.head(15)[roll_cols].to_string(index=False))

out(f"\n{'='*70}\ndocumented_need_no_plan ({len(documented_need_no_plan)} rows)\n{'='*70}")
np_cols = ["iso3", "country_name", "severity_score", "severity_category_lbl",
           "severity_trend", "crisis_drivers", "reliability"]
np_cols = [c for c in np_cols if c in documented_need_no_plan.columns]
out(documented_need_no_plan[np_cols].to_string(index=False))

# ============================================================
# 14) Example explanations from the top 5
# ============================================================
out(f"\n{'='*70}\nSample explanations (top 5)\n{'='*70}")
for _, r in ranked_gap_scores.head(5).iterrows():
    out(f"\n* {r['explanation']}")

# ============================================================
# 15) Save outputs
# ============================================================
try:
    ranked_gap_scores.to_parquet(OUTPUT_RANKED, index=False)
    country_rollup.to_parquet(OUTPUT_ROLLUP, index=False)
    documented_need_no_plan.to_parquet(OUTPUT_NOPLAN, index=False)
except Exception as e:
    out(f"\nParquet save warning: {e}")
    # fallback to CSV
    ranked_gap_scores.to_csv(OUTPUT_RANKED.replace(".parquet", ".csv"), index=False)
    country_rollup.to_csv(OUTPUT_ROLLUP.replace(".parquet", ".csv"), index=False)
    documented_need_no_plan.to_csv(OUTPUT_NOPLAN.replace(".parquet", ".csv"), index=False)

with open(OUTPUT_SENS, "w") as f:
    json.dump(sensitivity_report, f, indent=2)

with open(OUTPUT_TXT, "w") as f:
    f.write("\n".join(_buf))

print(f"\n\n>>> Output written:")
print(f"    text log:     {OUTPUT_TXT}")
print(f"    parquet:      {OUTPUT_RANKED}")
print(f"                  {OUTPUT_ROLLUP}")
print(f"                  {OUTPUT_NOPLAN}")
print(f"    sensitivity:  {OUTPUT_SENS}")
print(f"\n>>> To view all text output as plain text (easy to copy), run:")
print(f"    print(open('{OUTPUT_TXT}').read())")


GAP SCORE v1

Inputs OK: crises=239 rows, inform_country=68 rows
FTS cluster: 10635 rows
INFORM source: 202604-inform-severity-april-2026.xlsx
INFORM extra dims: 68 rows

ranked_gap_scores: 239 rows
country_rollup: 199 rows
documented_need_no_plan: 8 rows

Sensitivity results (top-10 stability + Tier 1 set stability):
  Baseline Tier 1 set (4 plans): ['COD|HCOD25|2025', 'LBN|RSYR25|2025', 'MMR|HMMR25b|2025', 'YEM|HYEM25|2025']
  severity_only       top10 mean 8.4/10 (FAIL)  |  Tier1 set: [4, 4, 4, 4, 4] / 4 (fully preserved)
  need_only           top10 mean 7.8/10 (FAIL)  |  Tier1 set: [4, 3, 4, 4, 3] / 4 (partial)
  coverage_only       top10 mean 7.8/10 (FAIL)  |  Tier1 set: [4, 4, 4, 3, 4] / 4 (partial)
  all_three           top10 mean 6.6/10 (FAIL)  |  Tier1 set: [3, 3, 4, 3, 3] / 4 (partial)

Top 15 ranked_gap_scores
iso3    plan_code                            plan_type  year  gap_score                     gap_tier  severity_norm  need_norm_absolute  coverage_gap_annualized  opera

In [0]:
# ============================================================
# GAP SCORE v1 — per GAP_SCORE_DESIGN.md spec
#
# Inputs (expected in memory):
#   - crises          (from loader_cell_v2.py)
#   - inform_country  (from inform_loader_v1.py)
#
# Outputs (in memory + on disk):
#   - ranked_gap_scores              (DataFrame + /tmp/gap_score_v1_ranked.parquet)
#   - country_rollup                 (DataFrame + /tmp/gap_score_v1_country_rollup.parquet)
#   - documented_need_no_plan        (DataFrame + /tmp/gap_score_v1_no_plan.parquet)
#   - sensitivity_report             (dict + /tmp/gap_score_v1_sensitivity.json)
#   - human-readable log             (/tmp/gap_score_v1_output.txt)
# ============================================================
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime
from pathlib import Path

VOLUME = "/Volumes/cmu_hackathon/common/unocha"
OUTPUT_TXT     = "/tmp/gap_score_v1_output.txt"
OUTPUT_RANKED  = "/tmp/gap_score_v1_ranked.parquet"
OUTPUT_ROLLUP  = "/tmp/gap_score_v1_country_rollup.parquet"
OUTPUT_NOPLAN  = "/tmp/gap_score_v1_no_plan.parquet"
OUTPUT_SENS    = "/tmp/gap_score_v1_sensitivity.json"

ANALYSIS_DATE = datetime(2026, 5, 18)

_buf = []
def out(*a, sep=" "):
    msg = sep.join(str(x) for x in a)
    _buf.append(msg)
    print(msg)

out("=" * 70)
out("GAP SCORE v1")
out("=" * 70)

# ============================================================
# 1) Country centroids (UN M49 source, embedded)
# ============================================================
CENTROIDS = {
    "AFG": (33.939, 67.710),    "AGO": (-11.203, 17.874),   "ARG": (-38.416, -63.617),
    "BDI": (-3.373, 29.919),    "BEN": (9.308, 2.316),      "BFA": (12.238, -1.562),
    "BGD": (23.685, 90.356),    "BGR": (42.734, 25.486),    "BOL": (-16.290, -63.589),
    "BRA": (-14.235, -51.925),  "CAF": (6.611, 20.939),     "CHL": (-35.675, -71.543),
    "CMR": (7.370, 12.355),     "COD": (-4.038, 21.759),    "COG": (-0.228, 15.828),
    "COL": (4.571, -74.297),    "CRI": (9.749, -83.753),    "CUW": (12.170, -68.990),
    "CZE": (49.817, 15.473),    "DJI": (11.825, 42.590),    "DOM": (18.736, -70.163),
    "ECU": (-1.831, -78.183),   "EGY": (26.821, 30.802),    "EST": (58.595, 25.014),
    "ETH": (9.145, 40.490),     "GTM": (15.783, -90.231),   "GUY": (4.860, -58.930),
    "HND": (15.200, -86.242),   "HTI": (18.971, -72.285),   "HUN": (47.162, 19.503),
    "IRN": (32.428, 53.688),    "JOR": (30.585, 36.238),    "KEN": (-0.024, 37.906),
    "LBN": (33.855, 35.862),    "LBY": (26.335, 17.228),    "LTU": (55.169, 23.881),
    "LVA": (56.880, 24.603),    "MDA": (47.412, 28.370),    "MEX": (23.635, -102.553),
    "MLI": (17.571, -3.996),    "MMR": (21.914, 95.956),    "MOZ": (-18.666, 35.530),
    "MWI": (-13.254, 34.302),   "NER": (17.608, 8.082),     "NGA": (9.082, 8.675),
    "PAK": (30.375, 69.345),    "PAN": (8.538, -80.782),    "PER": (-9.190, -75.015),
    "PHL": (12.880, 121.774),   "POL": (51.919, 19.145),    "PRY": (-23.443, -58.444),
    "PSE": (31.952, 35.233),    "ROU": (45.943, 24.967),    "RWA": (-1.940, 29.874),
    "SDN": (12.863, 30.218),    "SLV": (13.794, -88.897),   "SOM": (5.152, 46.200),
    "SSD": (6.877, 31.307),     "SVK": (48.669, 19.699),    "SYR": (34.802, 38.997),
    "TCD": (15.454, 18.732),    "TTO": (10.692, -61.223),   "TUR": (38.964, 35.243),
    "TZA": (-6.369, 34.889),    "UGA": (1.373, 32.290),     "UKR": (48.379, 31.166),
    "URY": (-32.523, -55.766),  "VEN": (6.424, -66.590),    "VNM": (14.058, 108.277),
    "YEM": (15.553, 48.516),    "ZMB": (-13.134, 27.849),   "ZWE": (-19.015, 29.155),
    "ABW": (12.521, -69.968),
}

# ============================================================
# 2) Validate required inputs are in memory
# ============================================================
try:
    _ = crises
except NameError:
    raise RuntimeError(
        "Required variable 'crises' not found. "
        "Run loader_cell_v2.py first, then re-run this cell."
    )
try:
    _ = inform_country
except NameError:
    raise RuntimeError(
        "Required variable 'inform_country' not found. "
        "Run inform_loader_v1.py first, then re-run this cell."
    )
out(f"\nInputs OK: crises={len(crises)} rows, inform_country={len(inform_country)} rows")

# ============================================================
# 3) Load FTS cluster (sector) data for hidden_sector_gap
# ============================================================
fts_cluster = pd.read_csv(
    f"{VOLUME}/fts_requirements_funding_globalcluster_global.csv", low_memory=False
)
fts_cluster = fts_cluster.dropna(subset=["code"]).copy()
fts_cluster = fts_cluster.rename(columns={
    "countryCode":   "iso3",
    "code":          "plan_code",
    "cluster":       "sector",
    "requirements":  "sector_requirements",
    "funding":       "sector_funding",
    "percentFunded": "sector_pct_funded",
})
fts_cluster["year"] = fts_cluster["year"].astype("Int64")
out(f"FTS cluster: {len(fts_cluster)} rows")

# ============================================================
# 4) Re-load latest INFORM file to fetch operating_env + society_safety
#    (inform_country from prior cell only kept a subset of columns)
# ============================================================
MONTH_NAMES = {"january":1,"february":2,"march":3,"april":4,"may":5,"june":6,
               "july":7,"august":8,"september":9,"october":10,"november":11,"december":12}

def parse_data_ym(name):
    n = name.lower()
    for mn, mv in MONTH_NAMES.items():
        m = re.search(rf"{mn}[-_ ](\d{{4}})", n)
        if m:
            return int(m.group(1)), mv
    return None, None

inform_xlsx = []
for f in sorted(Path(VOLUME).glob("*.xlsx")):
    n = f.name.lower()
    if "inform" not in n and "severity" not in n:
        continue
    y, mo = parse_data_ym(f.name)
    if y is None or y < 2021 or "(1)" in n or "mid_" in n or "mid-" in n:
        continue
    inform_xlsx.append({"path": f, "year": y, "month": mo})
inform_xlsx.sort(key=lambda r: (r["year"], r["month"]))
latest_inform = inform_xlsx[-1]
out(f"INFORM source: {latest_inform['path'].name}")

xl = pd.ExcelFile(latest_inform["path"])
candidates = [s for s in xl.sheet_names if s == "INFORM Severity - country"]
if not candidates:
    candidates = [s for s in xl.sheet_names if "FORM Severity - country" in s]
sheet = candidates[0]
inform_full = pd.read_excel(latest_inform["path"], sheet_name=sheet, header=1, skiprows=[2, 3])
if "TYPE OF CRISIS" in inform_full.columns and "DRIVERS" not in inform_full.columns:
    inform_full = inform_full.rename(columns={"TYPE OF CRISIS": "DRIVERS"})
inform_full = inform_full[inform_full["ISO3"].notna()].copy()
inform_extra = inform_full[["ISO3", "Operating environment", "Society and safety"]].rename(columns={
    "ISO3": "iso3",
    "Operating environment": "operating_env_score",
    "Society and safety":    "society_safety_score",
})
for c in ["operating_env_score", "society_safety_score"]:
    inform_extra[c] = pd.to_numeric(inform_extra[c], errors="coerce")
out(f"INFORM extra dims: {len(inform_extra)} rows")

# ============================================================
# 5) Build enriched table — Stage A from §8.3
# ============================================================
inform_join = inform_country[[
    "iso3", "country_name", "crisis_name", "crisis_drivers",
    "severity_score", "severity_category_lbl",
    "severity_trend", "reliability", "pin_score",
]].rename(columns={
    "severity_score":         "inform_severity",
    "severity_category_lbl":  "inform_severity_category",
    "severity_trend":         "inform_trend",
    "reliability":            "inform_reliability",
    "pin_score":              "inform_pin_score",
})

enriched = (crises
    .merge(inform_join, on="iso3", how="left")
    .merge(inform_extra, on="iso3", how="left"))

# country_name fallback if not in INFORM
enriched["country_name"] = enriched["country_name"].fillna(enriched["iso3"])

# centroids
enriched["latitude"]  = enriched["iso3"].map(lambda x: CENTROIDS.get(x, (None, None))[0])
enriched["longitude"] = enriched["iso3"].map(lambda x: CENTROIDS.get(x, (None, None))[1])

# hidden_sector_gap from fts_cluster
def worst_sector(g):
    if g["sector_pct_funded"].notna().sum() == 0:
        return pd.Series({"worst_sector_name": None, "worst_sector_pct_funded": None})
    idx = g["sector_pct_funded"].idxmin()
    return pd.Series({
        "worst_sector_name":       g.loc[idx, "sector"],
        "worst_sector_pct_funded": g.loc[idx, "sector_pct_funded"],
    })

sector_summary = (fts_cluster
    .groupby(["iso3", "plan_code", "year"], as_index=False)
    .apply(worst_sector)
    .reset_index(drop=True))

enriched = enriched.merge(sector_summary, on=["iso3", "plan_code", "year"], how="left")
enriched["hidden_sector_gap"] = (
    (enriched["worst_sector_pct_funded"].fillna(100) < 15)
    & (enriched["fts_pct_funded"].fillna(0) > 30)
)

# ============================================================
# 6) Compute components — Stage B from §8.3
# ============================================================
# severity_norm
def sev(row):
    if pd.notna(row["inform_severity"]):
        return row["inform_severity"] / 10, "INFORM"
    if pd.notna(row["inform_pin_score"]):
        return row["inform_pin_score"] / 10, "inform_pin_fallback"
    return 0.5, "default"
sev_data = enriched.apply(sev, axis=1)
enriched["severity_norm"]   = sev_data.map(lambda x: x[0])
enriched["severity_source"] = sev_data.map(lambda x: x[1])

# need_norm_absolute — PLAN-TYPE AWARE
# HRPs cover the full country response -> use HNO PIN when available
# Regional / Flash / Other plans cover a narrower sub-population -> use FTS requirements
# (don't inherit the country's total PIN onto narrow-scope plans like Syria 3RP slices
#  or Ethiopia regional refugee plans, which would over-inflate need)
HRP_TYPES = {"Humanitarian needs and response plan", "Humanitarian response plan"}

def need(row):
    is_full_country_plan = row.get("plan_type") in HRP_TYPES
    pin_ok = pd.notna(row["people_in_need"]) and row["people_in_need"] > 0
    req_ok = pd.notna(row["fts_requirements"]) and row["fts_requirements"] > 0
    if is_full_country_plan and pin_ok:
        return min(np.log10(max(row["people_in_need"], 1)) / 8, 1.0), "HNO_PIN"
    if req_ok:
        return min(np.log10(max(row["fts_requirements"] / 1e6, 1)) / 4, 1.0), "FTS_requirements_proxy"
    # Last resort: HRP with no PIN and no requirements -> 0
    if pin_ok:
        return min(np.log10(max(row["people_in_need"], 1)) / 8, 1.0), "HNO_PIN"
    return 0.0, "none"

need_data = enriched.apply(need, axis=1)
enriched["need_norm_absolute"] = need_data.map(lambda x: x[0])
enriched["need_source"]        = need_data.map(lambda x: x[1])

# need_norm_relative = PIN / population
enriched["need_norm_relative"] = np.where(
    (enriched["people_in_need"].notna()) & (enriched["population_hno"].fillna(0) > 0),
    enriched["people_in_need"] / enriched["population_hno"],
    np.nan,
)

# coverage_gap (raw + annualized) with year_progress guard
def days_elapsed(year):
    if year < ANALYSIS_DATE.year: return 365
    if year > ANALYSIS_DATE.year: return 0
    return (ANALYSIS_DATE - datetime(year, 1, 1)).days

def cov_gap(row):
    pct = row["fts_pct_funded"]
    yr  = int(row["year"])
    status = "closed" if yr < ANALYSIS_DATE.year else "in_progress" if yr == ANALYSIS_DATE.year else "future"
    if pd.isna(pct):
        return None, None, status
    pct = max(0, min(pct, 100))
    raw = 1 - pct / 100
    yp = days_elapsed(yr) / 365
    if status == "in_progress" and yp >= 0.20:
        expected = (pct / 100) / yp
        annualized = 1 - max(0, min(expected, 1.0))
    else:
        annualized = raw
    return raw, annualized, status

cov_data = enriched.apply(cov_gap, axis=1)
enriched["coverage_gap_raw"]        = cov_data.map(lambda x: x[0])
enriched["coverage_gap_annualized"] = cov_data.map(lambda x: x[1])
enriched["year_status"]             = cov_data.map(lambda x: x[2])

# ============================================================
# 7) Plan family + structural neglect — Stage C from §8.3
# ============================================================
def plan_family(code):
    if pd.isna(code):
        return None
    m = re.match(r"^(.+?)\d{2}[a-z]?$", str(code), re.IGNORECASE)
    return m.group(1) if m else str(code)

enriched["plan_family"] = enriched["plan_code"].map(plan_family)

# Coverage history lookup: (iso3, plan_family, year) -> max fts_pct_funded
hist = (enriched
    .dropna(subset=["plan_family", "fts_pct_funded"])
    .groupby(["iso3", "plan_family", "year"])["fts_pct_funded"]
    .max().to_dict())

def structural(row):
    if pd.isna(row["plan_family"]) or pd.isna(row["inform_severity"]):
        return False
    if row["inform_severity"] < 7:
        return False
    iso, fam, yr = row["iso3"], row["plan_family"], int(row["year"])
    priors = [hist.get((iso, fam, yr - 1)), hist.get((iso, fam, yr - 2))]
    valid = [c for c in priors if c is not None]
    if len(valid) < 2:
        return False
    return all(c < 50 for c in valid)

enriched["structural_neglect_flag"] = enriched.apply(structural, axis=1)

# coverage_inclusive_of_unattributed
def cov_incl(row):
    req = row["fts_requirements"]
    if pd.isna(req) or req == 0:
        return None
    fund = row["fts_funding"] if pd.notna(row["fts_funding"]) else 0
    unat = row["country_unattributed_funding"] if pd.notna(row["country_unattributed_funding"]) else 0
    return (fund + 0.5 * unat) / req

enriched["coverage_inclusive_of_unattributed"] = enriched.apply(cov_incl, axis=1)

# ============================================================
# 8) Gap score + confidence + explanation — Stage D from §8.3
# ============================================================
enriched["gap_score"] = (
    enriched["severity_norm"].fillna(0)
    * enriched["need_norm_absolute"].fillna(0)
    * enriched["coverage_gap_annualized"].fillna(0)
)

# Tier classification — avoid false precision by grouping scores into bands.
# Scores within ~0.10 of each other are statistically indistinguishable
# given measurement noise; tiers prevent over-interpretation of exact rank.
def gap_tier(score):
    if pd.isna(score):
        return "Tier 4 (insufficient data)"
    if score >= 0.55:
        return "Tier 1 (most overlooked)"
    if score >= 0.45:
        return "Tier 2 (severely overlooked)"
    if score >= 0.30:
        return "Tier 3 (underfunded)"
    return "Tier 4 (adequately resourced)"

enriched["gap_tier"] = enriched["gap_score"].apply(gap_tier)

def confidence_band(row):
    s = 0
    rel = row.get("inform_reliability")
    if rel in ("Very High", "High"): s += 2
    elif rel == "Medium":            s += 1
    if row.get("need_source") == "HNO_PIN":      s += 1
    if row.get("year_status") == "closed":       s += 1
    return "high" if s >= 3 else "medium" if s == 2 else "low"

enriched["confidence_band"] = enriched.apply(confidence_band, axis=1)

# Explanation
def explain(row):
    flags = []
    if row.get("structural_neglect_flag"):
        flags.append("Structural neglect: 2+ years <50% coverage.")
    if row.get("hidden_sector_gap"):
        flags.append(f"Hidden sector gap: {row['worst_sector_name']} at {row['worst_sector_pct_funded']:.0f}%.")
    op_env = row.get("operating_env_score")
    if pd.notna(op_env) and op_env >= 7.5:
        flags.append(f"Constrained operating environment: {op_env:.1f}/10 — funding alone may not enable delivery.")
    flags_str = " ".join(flags)
    pin_or_req = (
        f"{row['people_in_need']:,.0f} PIN"
        if pd.notna(row["people_in_need"])
        else f"${row['fts_requirements']/1e6:,.0f}M required"
        if pd.notna(row["fts_requirements"])
        else "need unknown"
    )
    sev_str = f"{row['inform_severity']:.1f}/10" if pd.notna(row.get("inform_severity")) else "n/a"
    trend = row.get("inform_trend", "?")
    cat = row.get("inform_severity_category", "?")
    pct = f"{row['fts_pct_funded']:.0f}%" if pd.notna(row.get("fts_pct_funded")) else "n/a"
    return (
        f"{row['country_name']} {row['plan_code']} ({row['plan_type']}) — "
        f"severity {sev_str} ({cat}, trend {trend}), "
        f"need from {row['need_source']} ({pin_or_req}), "
        f"coverage {pct} ({row['year_status']}). "
        f"Gap score {row['gap_score']:.2f} ({row['gap_tier']}). "
        f"Confidence: {row['confidence_band']}. {flags_str}"
    ).strip()

enriched["explanation"] = enriched.apply(explain, axis=1)

# ============================================================
# 9) Table 1: ranked_gap_scores — Stage E
# ============================================================
output_cols = [
    "iso3", "country_name", "plan_code", "plan_type", "year",
    "gap_score", "gap_tier",
    "severity_norm", "need_norm_absolute", "coverage_gap_annualized",
    "need_norm_relative", "coverage_gap_raw",
    "severity_source", "need_source", "hno_source_year", "year_status",
    "inform_reliability", "confidence_band",
    "inform_trend",
    "operating_env_score", "society_safety_score",
    "hidden_sector_gap", "worst_sector_name", "worst_sector_pct_funded",
    "structural_neglect_flag", "plan_family",
    "fts_requirements", "fts_funding", "fts_pct_funded",
    "coverage_inclusive_of_unattributed",
    "people_in_need", "population_hno",
    "country_unattributed_funding",
    "latitude", "longitude",
    "explanation",
]
ranked_gap_scores = (enriched[output_cols]
    .sort_values("gap_score", ascending=False)
    .reset_index(drop=True))
out(f"\nranked_gap_scores: {len(ranked_gap_scores)} rows")

# ============================================================
# 10) Table 2: country_rollup
# ============================================================
CB_ORDER = {"low": 0, "medium": 1, "high": 2}
CB_INV   = {v: k for k, v in CB_ORDER.items()}

def country_agg(g):
    sum_req  = g["fts_requirements"].fillna(0).sum()
    sum_fund = g["fts_funding"].fillna(0).sum()
    underf   = (g["fts_pct_funded"].fillna(100) < 50).sum()
    min_cb   = min((CB_ORDER.get(c, 1) for c in g["confidence_band"]), default=1)
    worst_idx = g["fts_pct_funded"].idxmin() if g["fts_pct_funded"].notna().any() else g.index[0]
    return pd.Series({
        "country_name":         g["country_name"].iloc[0],
        "severity_score":       g["inform_severity"].iloc[0] if pd.notna(g["inform_severity"].iloc[0]) else None,
        "severity_trend":       g["inform_trend"].iloc[0],
        "n_plans":              len(g),
        "n_plans_underfunded":  int(underf),
        "sum_fts_requirements": float(sum_req),
        "sum_fts_funding":      float(sum_fund),
        "sum_funding_gap_usd":  float(sum_req - sum_fund),
        "country_coverage_pct": (sum_fund / sum_req * 100) if sum_req > 0 else None,
        "worst_plan_coverage_pct": g["fts_pct_funded"].min(),
        "worst_plan_code":      g.loc[worst_idx, "plan_code"],
        "max_gap_score":        g["gap_score"].max(),
        "mean_gap_score":       g["gap_score"].mean(),
        "gap_tier":             gap_tier(g["gap_score"].max()),
        "operating_env_score":  g["operating_env_score"].iloc[0] if pd.notna(g["operating_env_score"].iloc[0]) else None,
        "society_safety_score": g["society_safety_score"].iloc[0] if pd.notna(g["society_safety_score"].iloc[0]) else None,
        "any_structural_neglect": bool(g["structural_neglect_flag"].any()),
        "any_hidden_sector_gap":  bool(g["hidden_sector_gap"].any()),
        "confidence_band":      CB_INV[min_cb],
        "latitude":             g["latitude"].iloc[0],
        "longitude":            g["longitude"].iloc[0],
    })

country_rollup = (enriched
    .groupby(["iso3", "year"], as_index=False)
    .apply(country_agg)
    .reset_index(drop=True)
    .sort_values("sum_funding_gap_usd", ascending=False)
    .reset_index(drop=True))
out(f"country_rollup: {len(country_rollup)} rows")

# ============================================================
# 11) Table 3: documented_need_no_plan
# ============================================================
countries_with_plan = set(
    enriched[
        enriched["plan_code"].notna()
        & (enriched["fts_requirements"].fillna(0) > 0)
        & enriched["year"].between(2024, 2026)
    ]["iso3"].unique()
)

documented_need_no_plan = inform_country[
    (inform_country["severity_score"] >= 5)
    & (~inform_country["iso3"].isin(countries_with_plan))
].copy()
documented_need_no_plan["latitude"]  = documented_need_no_plan["iso3"].map(lambda x: CENTROIDS.get(x, (None, None))[0])
documented_need_no_plan["longitude"] = documented_need_no_plan["iso3"].map(lambda x: CENTROIDS.get(x, (None, None))[1])
documented_need_no_plan = (documented_need_no_plan
    .sort_values("severity_score", ascending=False)
    .reset_index(drop=True))
out(f"documented_need_no_plan: {len(documented_need_no_plan)} rows")

# ============================================================
# 12) Sensitivity analysis — Stage F
# ============================================================
np.random.seed(42)
baseline_top10 = set(
    ranked_gap_scores.head(10)
    .apply(lambda r: f"{r['iso3']}|{r['plan_code']}|{r['year']}", axis=1)
)

def perturb(scenario, cols, n_iter=5):
    rates = []
    for _ in range(n_iter):
        p = enriched.copy()
        for c in cols:
            noise = np.random.uniform(-0.05, 0.05, size=len(p))
            p[c] = (p[c].fillna(0) + noise).clip(0, 1)
        p["gap_perturbed"] = (
            p["severity_norm"].fillna(0)
            * p["need_norm_absolute"].fillna(0)
            * p["coverage_gap_annualized"].fillna(0)
        )
        new_top10 = set(
            p.nlargest(10, "gap_perturbed")
            .apply(lambda r: f"{r['iso3']}|{r['plan_code']}|{r['year']}", axis=1)
        )
        rates.append(len(baseline_top10 & new_top10))
    return {
        "scenario": scenario,
        "preservation_rates": rates,
        "mean_preserved_in_top_10": float(np.mean(rates)),
        "passes_8_of_10_threshold": all(r >= 8 for r in rates),
    }

sensitivity_report = {
    "method": "absolute +/- 0.05 uniform noise on [0,1] components",
    "n_iter_per_scenario": 5,
    "scenarios": [
        perturb("severity_only", ["severity_norm"]),
        perturb("need_only",     ["need_norm_absolute"]),
        perturb("coverage_only", ["coverage_gap_annualized"]),
        perturb("all_three",     ["severity_norm", "need_norm_absolute", "coverage_gap_annualized"]),
    ],
}

out(f"\nSensitivity results (top-10 stability):")
for s in sensitivity_report["scenarios"]:
    out(f"  {s['scenario']:18s}  mean {s['mean_preserved_in_top_10']:.1f}/10  "
        f"rates={s['preservation_rates']}  "
        f"{'PASS' if s['passes_8_of_10_threshold'] else 'FAIL'}")

# ============================================================
# 13) Print headline outputs
# ============================================================
out(f"\n{'='*70}\nTop 15 ranked_gap_scores\n{'='*70}")
top_cols = ["iso3", "plan_code", "plan_type", "year", "gap_score", "gap_tier",
            "severity_norm", "need_norm_absolute", "coverage_gap_annualized",
            "operating_env_score", "confidence_band",
            "structural_neglect_flag", "hidden_sector_gap"]
out(ranked_gap_scores.head(15)[top_cols].to_string(index=False))

out(f"\n{'='*70}\nTier distribution\n{'='*70}")
out(ranked_gap_scores["gap_tier"].value_counts().sort_index().to_string())

out(f"\n{'='*70}\nTop 15 country_rollup (by sum_funding_gap_usd)\n{'='*70}")
roll_cols = ["iso3", "country_name", "year", "n_plans", "n_plans_underfunded",
             "sum_funding_gap_usd", "country_coverage_pct", "max_gap_score",
             "gap_tier", "operating_env_score", "society_safety_score",
             "any_structural_neglect", "confidence_band"]
out(country_rollup.head(15)[roll_cols].to_string(index=False))

out(f"\n{'='*70}\ndocumented_need_no_plan ({len(documented_need_no_plan)} rows)\n{'='*70}")
np_cols = ["iso3", "country_name", "severity_score", "severity_category_lbl",
           "severity_trend", "crisis_drivers", "reliability"]
np_cols = [c for c in np_cols if c in documented_need_no_plan.columns]
out(documented_need_no_plan[np_cols].to_string(index=False))

# ============================================================
# 14) Example explanations from the top 5
# ============================================================
out(f"\n{'='*70}\nSample explanations (top 5)\n{'='*70}")
for _, r in ranked_gap_scores.head(5).iterrows():
    out(f"\n* {r['explanation']}")

# ============================================================
# 15) Save outputs
# ============================================================
try:
    ranked_gap_scores.to_parquet(OUTPUT_RANKED, index=False)
    country_rollup.to_parquet(OUTPUT_ROLLUP, index=False)
    documented_need_no_plan.to_parquet(OUTPUT_NOPLAN, index=False)
except Exception as e:
    out(f"\nParquet save warning: {e}")
    # fallback to CSV
    ranked_gap_scores.to_csv(OUTPUT_RANKED.replace(".parquet", ".csv"), index=False)
    country_rollup.to_csv(OUTPUT_ROLLUP.replace(".parquet", ".csv"), index=False)
    documented_need_no_plan.to_csv(OUTPUT_NOPLAN.replace(".parquet", ".csv"), index=False)

with open(OUTPUT_SENS, "w") as f:
    json.dump(sensitivity_report, f, indent=2)

with open(OUTPUT_TXT, "w") as f:
    f.write("\n".join(_buf))

print(f"\n\n>>> Output written:")
print(f"    text log:     {OUTPUT_TXT}")
print(f"    parquet:      {OUTPUT_RANKED}")
print(f"                  {OUTPUT_ROLLUP}")
print(f"                  {OUTPUT_NOPLAN}")
print(f"    sensitivity:  {OUTPUT_SENS}")
print(f"\n>>> To view all text output as plain text (easy to copy), run:")
print(f"    print(open('{OUTPUT_TXT}').read())")


GAP SCORE v1

Inputs OK: crises=239 rows, inform_country=68 rows
FTS cluster: 10635 rows
INFORM source: 202604-inform-severity-april-2026.xlsx
INFORM extra dims: 68 rows

ranked_gap_scores: 239 rows
country_rollup: 199 rows
documented_need_no_plan: 8 rows

Sensitivity results (top-10 stability):
  severity_only       mean 8.4/10  rates=[7, 8, 8, 10, 9]  FAIL
  need_only           mean 7.8/10  rates=[8, 8, 7, 8, 8]  FAIL
  coverage_only       mean 7.8/10  rates=[9, 5, 7, 9, 9]  FAIL
  all_three           mean 6.6/10  rates=[7, 6, 6, 7, 7]  FAIL

Top 15 ranked_gap_scores
iso3    plan_code                            plan_type  year  gap_score                     gap_tier  severity_norm  need_norm_absolute  coverage_gap_annualized  operating_env_score confidence_band  structural_neglect_flag  hidden_sector_gap
 LBN       RSYR25               Regional response plan  2025   0.608909     Tier 1 (most overlooked)           0.77            0.869001                 0.910000                  8.5 

In [0]:
# Verify pipeline state — useful quick check between stages
try:
    print(f"crises:                  {crises.shape}")
    print(f"inform_country:          {inform_country.shape}")
    print(f"ranked_gap_scores:       {ranked_gap_scores.shape}")
    print(f"country_rollup:          {country_rollup.shape}")
    print(f"documented_need_no_plan: {documented_need_no_plan.shape}")
    print(f"\nTier distribution:")
    print(ranked_gap_scores["gap_tier"].value_counts().sort_index().to_string())
    print(f"\nTier 1 plans:")
    cols = ["iso3", "plan_code", "plan_type", "gap_score"]
    print(ranked_gap_scores[ranked_gap_scores["gap_tier"] == "Tier 1 (most overlooked)"][cols].to_string(index=False))
except NameError as e:
    print(f"Pipeline not fully run: {e}")


crises:                  (239, 14)
inform_country:          (68, 16)
ranked_gap_scores:       (239, 36)
country_rollup:          (199, 23)
documented_need_no_plan: (8, 18)

Tier distribution:
Tier 1 (most overlooked)           4
Tier 2 (severely overlooked)      22
Tier 3 (underfunded)              46
Tier 4 (adequately resourced)    167

Tier 1 plans:
iso3 plan_code                            plan_type  gap_score
 LBN    RSYR25               Regional response plan   0.608909
 COD    HCOD25 Humanitarian needs and response plan   0.601746
 YEM    HYEM25 Humanitarian needs and response plan   0.595304
 MMR   HMMR25b Humanitarian needs and response plan   0.567505


In [0]:
# ============================================================
# VERIFICATION TEST — does the pipeline output match the writeup claims?
# ============================================================
# Run this AFTER all the pipeline cells have completed.
# Expects in scope: crises, inform_country, ranked_gap_scores,
#                   country_rollup, documented_need_no_plan
#
# Prints a pass/fail report for every documented finding.
# ============================================================
import os
import pandas as pd

results = []  # list of (status, label, expected, actual, note)

def check(label, condition, expected, actual, note=""):
    status = "PASS" if condition else "FAIL"
    results.append((status, label, str(expected), str(actual), note))

# ============================================================
# 1) DataFrame shapes
# ============================================================
check("crises rows",
      crises.shape[0] == 239,
      239, crises.shape[0])

check("crises columns",
      crises.shape[1] == 14,
      14, crises.shape[1])

check("inform_country rows",
      inform_country.shape[0] == 68,
      68, inform_country.shape[0])

check("ranked_gap_scores rows",
      ranked_gap_scores.shape[0] == 239,
      239, ranked_gap_scores.shape[0])

check("country_rollup rows",
      country_rollup.shape[0] == 199,
      199, country_rollup.shape[0])

check("documented_need_no_plan rows",
      documented_need_no_plan.shape[0] == 8,
      8, documented_need_no_plan.shape[0])

# ============================================================
# 2) Tier distribution
# ============================================================
tier_counts = ranked_gap_scores["gap_tier"].value_counts().to_dict()

check("Tier 1 count = 4",
      tier_counts.get("Tier 1 (most overlooked)") == 4,
      4, tier_counts.get("Tier 1 (most overlooked)"))

check("Tier 2 count = 22",
      tier_counts.get("Tier 2 (severely overlooked)") == 22,
      22, tier_counts.get("Tier 2 (severely overlooked)"))

check("Tier 3 count = 46",
      tier_counts.get("Tier 3 (underfunded)") == 46,
      46, tier_counts.get("Tier 3 (underfunded)"))

check("Tier 4 count = 167",
      tier_counts.get("Tier 4 (adequately resourced)") == 167,
      167, tier_counts.get("Tier 4 (adequately resourced)"))

# ============================================================
# 3) Tier 1 plans — the headline
# ============================================================
tier1 = ranked_gap_scores[
    ranked_gap_scores["gap_tier"] == "Tier 1 (most overlooked)"
].sort_values("gap_score", ascending=False)
tier1_codes = set(tier1["plan_code"].tolist())

check("Tier 1 = exactly 4 plans",
      len(tier1) == 4,
      4, len(tier1))

check("LBN RSYR25 (Syria 3RP) in Tier 1",
      "RSYR25" in tier1_codes,
      "in Tier 1", "found" if "RSYR25" in tier1_codes else "NOT FOUND")

check("COD HCOD25 (DRC) in Tier 1",
      "HCOD25" in tier1_codes,
      "in Tier 1", "found" if "HCOD25" in tier1_codes else "NOT FOUND")

check("YEM HYEM25 (Yemen) in Tier 1",
      "HYEM25" in tier1_codes,
      "in Tier 1", "found" if "HYEM25" in tier1_codes else "NOT FOUND")

check("MMR HMMR25b (Myanmar) in Tier 1",
      "HMMR25b" in tier1_codes,
      "in Tier 1", "found" if "HMMR25b" in tier1_codes else "NOT FOUND")

# Top plan should be Lebanon-RSYR25
top1 = tier1.iloc[0]
check("#1 plan is LBN RSYR25",
      top1["plan_code"] == "RSYR25" and top1["iso3"] == "LBN",
      "LBN RSYR25", f"{top1['iso3']} {top1['plan_code']}")

# ============================================================
# 4) Sudan paradox — severity 9.6 but Tier 2 (not Tier 1)
# ============================================================
sdn25 = ranked_gap_scores[
    (ranked_gap_scores["iso3"] == "SDN") &
    (ranked_gap_scores["plan_code"] == "HSDN25")
]
if len(sdn25) == 1:
    sdn_row = sdn25.iloc[0]
    check("Sudan HSDN25 severity = 9.6 (highest in dataset)",
          abs(sdn_row["severity_norm"] - 0.96) < 0.01,
          0.96, round(sdn_row["severity_norm"], 3))
    check("Sudan HSDN25 ranks Tier 2 (not Tier 1)",
          sdn_row["gap_tier"] == "Tier 2 (severely overlooked)",
          "Tier 2 (severely overlooked)", sdn_row["gap_tier"])
else:
    check("Sudan HSDN25 present", False, 1, len(sdn25), "row missing")

# ============================================================
# 5) Palestine: largest absolute gap + op_env = 10.0
# ============================================================
pse_rollup = country_rollup[country_rollup["iso3"] == "PSE"]
if len(pse_rollup) > 0:
    pse_2026 = pse_rollup[pse_rollup["year"] == 2026]
    if len(pse_2026) == 1:
        pse_row = pse_2026.iloc[0]
        # ~3.58B raw funding gap
        check("Palestine 2026 absolute gap ≈ $3.58B",
              abs(pse_row["sum_funding_gap_usd"] - 3.58e9) < 0.05e9,
              "≈ 3.58e9", f"{pse_row['sum_funding_gap_usd']:.2e}")
        check("Palestine 2026 op_env = 10.0",
              pse_row["operating_env_score"] == 10.0,
              10.0, pse_row["operating_env_score"])

# Palestine #1 by absolute gap globally
top_by_gap = country_rollup.sort_values("sum_funding_gap_usd", ascending=False).iloc[0]
check("Palestine 2026 is #1 by absolute funding gap globally",
      top_by_gap["iso3"] == "PSE" and top_by_gap["year"] == 2026,
      "PSE 2026", f"{top_by_gap['iso3']} {top_by_gap['year']}")

# ============================================================
# 6) Palestine has no HNO in any year (need_source = FTS_proxy)
# ============================================================
pse_plans = ranked_gap_scores[ranked_gap_scores["iso3"] == "PSE"]
if len(pse_plans) > 0:
    pse_need_sources = set(pse_plans["need_source"].dropna().unique())
    check("Palestine uses FTS_requirements_proxy (no HNO PIN)",
          pse_need_sources == {"FTS_requirements_proxy"},
          "{'FTS_requirements_proxy'}", str(pse_need_sources))

# ============================================================
# 7) Mauritania in documented_need_no_plan
# ============================================================
mrt = documented_need_no_plan[documented_need_no_plan["iso3"] == "MRT"]
check("Mauritania in documented_need_no_plan",
      len(mrt) == 1,
      1, len(mrt))

if len(mrt) == 1:
    mrt_row = mrt.iloc[0]
    check("Mauritania severity ≈ 6.1",
          abs(mrt_row["severity_score"] - 6.1) < 0.1,
          6.1, round(mrt_row["severity_score"], 2))
    check("Mauritania trend = Increasing",
          mrt_row["severity_trend"] == "Increasing",
          "Increasing", mrt_row["severity_trend"])

# ============================================================
# 8) Afghanistan trajectory (52% -> 42% -> 15%)
# ============================================================
for year, exp_pct, label in [(2024, 52, "AFG 2024 ≈ 52%"),
                              (2025, 42, "AFG 2025 ≈ 42%"),
                              (2026, 15, "AFG 2026 ≈ 15%")]:
    afg = ranked_gap_scores[
        (ranked_gap_scores["iso3"] == "AFG") &
        (ranked_gap_scores["plan_code"].str.startswith("HAFG")) &
        (ranked_gap_scores["year"] == year)
    ]
    if len(afg) == 1:
        actual_pct = afg.iloc[0]["fts_pct_funded"]
        check(label,
              abs(actual_pct - exp_pct) < 2,
              f"≈ {exp_pct}%", f"{actual_pct:.0f}%")

# ============================================================
# 9) Hidden sector gaps fire for SDN and ETH HRPs
# ============================================================
hidden_gaps = ranked_gap_scores[ranked_gap_scores["hidden_sector_gap"] == True]
hidden_codes = set(hidden_gaps["plan_code"].tolist())

check("hidden_sector_gap fires for SDN HSDN25",
      "HSDN25" in hidden_codes,
      "fires", "fires" if "HSDN25" in hidden_codes else "DOES NOT FIRE")
check("hidden_sector_gap fires for ETH HETH24",
      "HETH24" in hidden_codes,
      "fires", "fires" if "HETH24" in hidden_codes else "DOES NOT FIRE")

# ============================================================
# 10) Durable output files exist
# ============================================================
try:
    current_user = spark.sql("SELECT current_user()").collect()[0][0]
    workspace_dir = f"/Workspace/Users/{current_user}/asli_gulcur_submission"
    expected_files = [
        "gap_score_v1_output.txt",
        "gap_score_v1_ranked.parquet",
        "gap_score_v1_country_rollup.parquet",
        "gap_score_v1_no_plan.parquet",
        "gap_score_v1_sensitivity.json",
    ]
    for fname in expected_files:
        path = f"{workspace_dir}/{fname}"
        exists = os.path.exists(path)
        check(f"durable output exists: {fname}",
              exists,
              "exists", "exists" if exists else "MISSING", note=path)
except Exception as e:
    check("durable storage check", False, "no error", str(e))

# ============================================================
# Print report
# ============================================================
print("=" * 90)
print(f"VERIFICATION REPORT  —  {len(results)} checks")
print("=" * 90)
pass_count = sum(1 for r in results if r[0] == "PASS")
fail_count = sum(1 for r in results if r[0] == "FAIL")

for status, label, expected, actual, note in results:
    marker = "✓" if status == "PASS" else "✗"
    extra  = f"  ({note})" if note else ""
    print(f"  {marker} [{status}]  {label}")
    if status == "FAIL":
        print(f"           expected: {expected}")
        print(f"           actual:   {actual}{extra}")

print()
print("=" * 90)
print(f"  PASSED: {pass_count} / {len(results)}    FAILED: {fail_count}")
print("=" * 90)
if fail_count == 0:
    print("\n  ✓ ALL CHECKS PASSED — pipeline output matches every documented finding.")
else:
    print(f"\n  ✗ {fail_count} CHECK(S) FAILED — review the actual values above")
    print("    and decide if the writeup needs an update OR the pipeline does.")


VERIFICATION REPORT  —  35 checks
  ✓ [PASS]  crises rows
  ✓ [PASS]  crises columns
  ✓ [PASS]  inform_country rows
  ✓ [PASS]  ranked_gap_scores rows
  ✓ [PASS]  country_rollup rows
  ✓ [PASS]  documented_need_no_plan rows
  ✓ [PASS]  Tier 1 count = 4
  ✓ [PASS]  Tier 2 count = 22
  ✓ [PASS]  Tier 3 count = 46
  ✓ [PASS]  Tier 4 count = 167
  ✓ [PASS]  Tier 1 = exactly 4 plans
  ✓ [PASS]  LBN RSYR25 (Syria 3RP) in Tier 1
  ✓ [PASS]  COD HCOD25 (DRC) in Tier 1
  ✓ [PASS]  YEM HYEM25 (Yemen) in Tier 1
  ✓ [PASS]  MMR HMMR25b (Myanmar) in Tier 1
  ✓ [PASS]  #1 plan is LBN RSYR25
  ✓ [PASS]  Sudan HSDN25 severity = 9.6 (highest in dataset)
  ✓ [PASS]  Sudan HSDN25 ranks Tier 2 (not Tier 1)
  ✓ [PASS]  Palestine 2026 absolute gap ≈ $3.58B
  ✓ [PASS]  Palestine 2026 op_env = 10.0
  ✓ [PASS]  Palestine 2026 is #1 by absolute funding gap globally
  ✓ [PASS]  Palestine uses FTS_requirements_proxy (no HNO PIN)
  ✓ [PASS]  Mauritania in documented_need_no_plan
  ✓ [PASS]  Mauritania severity ≈ 

## Stage 4 — Bonus task: multi-year INFORM time series + temporal signals

### What this stage does (answering the PDF's bonus question)

The PDF's bonus question:

> *"How should ranking change when a crisis has been underfunded for multiple consecutive years versus one that is newly underfunded? How can you represent structural issues differently from acute emergencies?"*

This stage answers it with two scripts:

1. **`inform_timeseries_v1.py`** — loads 62 deduped monthly INFORM Severity files (Jan 2021 – Apr 2026), 4,927 observations across 108 countries. Computes four scale-invariant per-country signals.
2. **`merge_temporal_signals.py`** — joins the per-country signals into `ranked_gap_scores` and `country_rollup`, replacing the broken v1 `structural_neglect_flag` (which never fired) with the working v2 (137 plan-rows fire).

### The four temporal signals

All use INFORM **categorical labels** (Very Low → Very High), not raw numeric scores, so they're robust across the January 2026 scale change (see below).

| Signal | Definition | Use |
|---|---|---|
| `months_in_high_plus_24mo` | Count of months in "High" or "Very High" in last 24 | Chronic-severity baseline |
| `months_very_high_24mo` | Same, restricted to "Very High" | Acute chronic-severity |
| `months_increasing_12mo` | Count of months with `severity_trend == "Increasing"` in last 12 | Rising-risk signal |
| `structural_neglect_v2` | True when `months_in_high_plus_24mo ≥ 18` | Replaces broken v1 flag |

### The discovery that almost broke us

While building the time series, we observed **median severity jumping from 3.10 to 6.15 between December 2025 and January 2026** — a ratio of 1.98×. Our first instinct was a fast-rising crisis; our second was a methodology change. We confirmed the second: **INFORM rescaled from 1–5 to 1–10 in January 2026**. The first version of the temporal signals used numeric thresholds and silently produced wrong results. We rebuilt every signal on the categorical labels (which are scale-invariant) and verified against expected results. This is the data due-diligence finding inside the bonus task.

### Why a parallel signal, not a multiplier

The PDF suggested:

```
adjusted_gap_score = gap_score × (1 + 0.1 × n_years_below_50pct_coverage)
```

We considered this and chose the parallel-signal approach instead. A multiplier silently inflates the rank for chronic cases, obscuring the decision logic. Showing chronic-neglect as a separate column makes the signal interpretable and overridable — matching the design philosophy of `operating_env_score`.

### Three findings the bonus task produced

1. **Angola is the standout rising-risk** — 10 of last 12 months Increasing, currently in High category, currently absent from any top tier. A funding-gap analysis alone would miss it.
2. **`structural_neglect_v2` fires for 37 countries; 36 of 37 are already in our gap-score ranking** — only DPRK (no FTS plan in our window) is invisible. This validates the FTS-as-backbone architecture: the chronic-severity signal isn't surfacing new countries, it's confirming the existing ranking is comprehensive.
3. **All four Tier 1 plans flag `structural_neglect_flag = True`**, with COD/YEM at 22-of-22 months Very High (the ceiling) and MMR at 21-of-22. Sudan (Tier 2 by gap_score) shows the same 22-of-22 Very-High pattern. The chronicity signal cross-validates the funding-gap signal.

### Output of this stage

- `inform_timeseries` — long-format DataFrame, **4,927 rows × 9 cols** (Jan 2021 – Apr 2026, 108 countries)
- `inform_country_signals` — per-country aggregate, **108 rows × 13 cols**
- `ranked_gap_scores` updated to **239 × 40** with v2 flag + four temporal columns merged
- `country_rollup` updated to **199 × 27** with `any_structural_neglect` + temporal columns


In [0]:

# ============================================================
# INFORM Severity time series loader — v2 (bonus task)
#
# Builds:
#   1. inform_timeseries: long-format DataFrame
#      (iso3, year, month, severity_score, severity_category_lbl, trend, reliability,
#       drivers, source_file, date_idx)
#   2. inform_country_signals: per-country temporal signals
#      (iso3, months_in_high_plus_24mo, months_very_high_24mo,
#       months_increasing_12mo, months_decreasing_12mo, structural_neglect_v2)
#
# IMPORTANT methodology note (discovered during v1 run):
#   INFORM changed its severity scale around January 2026, from a 1–5 range
#   to a 1–10 range (values approximately doubled overnight). This makes raw
#   numerical comparisons across the scale boundary unsafe. This v2 of the
#   loader uses the CATEGORICAL label (Very Low / Low / Medium / High / Very High)
#   for chronic-neglect detection, since labels are scale-invariant.
#
# Output: /tmp/inform_timeseries_output.txt (human-readable summary)
#         /tmp/inform_timeseries.parquet
#         /tmp/inform_country_signals.parquet
# ============================================================
import pandas as pd
import numpy as np
import re
import time
from pathlib import Path

VOLUME = "/Volumes/cmu_hackathon/common/unocha"
OUTPUT_TXT     = "/tmp/inform_timeseries_output.txt"
OUTPUT_TS_PQT  = "/tmp/inform_timeseries.parquet"
OUTPUT_SIG_PQT = "/tmp/inform_country_signals.parquet"

# Categorical thresholds (scale-invariant)
HIGH_PLUS_CATEGORIES = {"High", "Very High"}
VERY_HIGH_CATEGORY   = {"Very High"}

# Window sizes
WINDOW_24 = 24
WINDOW_12 = 12

# ============================================================
# Buffered output helper
# ============================================================
_buf = []
def out(*a, sep=" "):
    msg = sep.join(str(x) for x in a)
    _buf.append(msg)
    print(msg)

out("=" * 70)
out("INFORM SEVERITY TIME SERIES — bonus task (v2 — scale-invariant)")
out("=" * 70)

# ============================================================
# 1) Discover and dedupe INFORM files
# ============================================================
MONTH_NAMES = {
    "january": 1, "february": 2, "march": 3,    "april": 4,
    "may": 5,     "june": 6,     "july": 7,     "august": 8,
    "september": 9,"october": 10, "november": 11,"december": 12,
}

def parse_data_year_month(name):
    n = name.lower()
    for mname, mnum in MONTH_NAMES.items():
        m = re.search(rf"{mname}[-_ ](\d{{4}})", n)
        if m:
            return int(m.group(1)), mnum
    return None, None

raw_files = []
for f in sorted(Path(VOLUME).glob("*.xlsx")):
    name = f.name.lower()
    if "inform" not in name and "severity" not in name:
        continue
    y, m = parse_data_year_month(f.name)
    if y is None:
        continue
    if y < 2021:
        continue
    if "(1)" in name:
        continue
    if "mid_" in name or "mid-" in name:
        continue
    raw_files.append({"path": f, "name": f.name, "year": y, "month": m})

# Dedupe by (year, month) — prefer the shorter filename (no suffix like "-1")
raw_files.sort(key=lambda r: (r["year"], r["month"], len(r["name"])))
inform_files = []
seen = set()
duplicates_dropped = []
for f in raw_files:
    key = (f["year"], f["month"])
    if key in seen:
        duplicates_dropped.append(f["name"])
        continue
    seen.add(key)
    inform_files.append(f)

out(f"\nDiscovered {len(raw_files)} candidate files; deduped to {len(inform_files)} (one per year-month)")
if duplicates_dropped:
    out(f"  Duplicates dropped ({len(duplicates_dropped)}):")
    for n in duplicates_dropped:
        out(f"    - {n}")
out(f"  Date range: {inform_files[0]['year']}-{inform_files[0]['month']:02d}  ->  "
    f"{inform_files[-1]['year']}-{inform_files[-1]['month']:02d}")

# ============================================================
# 2) Country-sheet reader (defensive against schema drift)
# ============================================================
def read_country_sheet(path):
    try:
        xl = pd.ExcelFile(path)
    except Exception as e:
        return None, f"Excel open failed: {e}"

    candidates = [s for s in xl.sheet_names if s == "INFORM Severity - country"]
    if not candidates:
        candidates = [s for s in xl.sheet_names if "FORM Severity - country" in s]
    if not candidates:
        candidates = [s for s in xl.sheet_names
                      if "country" in s.lower() and "severity" in s.lower()]
    if not candidates:
        return None, "no country sheet found"

    sheet = candidates[0]
    try:
        raw = pd.read_excel(path, sheet_name=sheet, header=1, skiprows=[2, 3])
    except Exception as e:
        return None, f"read_excel failed: {e}"

    if "TYPE OF CRISIS" in raw.columns and "DRIVERS" not in raw.columns:
        raw = raw.rename(columns={"TYPE OF CRISIS": "DRIVERS"})

    if "ISO3" not in raw.columns or "INFORM Severity Index" not in raw.columns:
        return None, "missing required columns"

    df = raw[raw["ISO3"].notna()].copy()
    keep = {
        "ISO3":                       "iso3",
        "COUNTRY":                    "country_name",
        "DRIVERS":                    "crisis_drivers",
        "INFORM Severity Index":      "severity_score",
        "INFORM Severity category.1": "severity_category_lbl",
        "Trend (last 3 months)":      "severity_trend",
        "Reliability":                "reliability",
    }
    available = {k: v for k, v in keep.items() if k in df.columns}
    df = df[list(available.keys())].rename(columns=available)
    df["severity_score"] = pd.to_numeric(df["severity_score"], errors="coerce")
    df = df[df["severity_score"].notna()]
    # Fallback: derive category from numeric score if category column missing
    if "severity_category_lbl" not in df.columns:
        def derive_cat(s):
            # Map either scale (1-5 or 1-10) to category via thresholds.
            # Old scale: <2 Very Low, 2-3 Low, 3-3.5 Medium, 3.5-4.5 High, >=4.5 Very High
            # New scale: <2 Very Low, 2-4 Low, 4-6 Medium, 6-8 High, >=8 Very High
            if pd.isna(s): return None
            if s >= 8 or s >= 4.5: return "Very High"
            if s >= 6 or s >= 3.5: return "High"
            if s >= 4 or s >= 3:   return "Medium"
            if s >= 2:             return "Low"
            return "Very Low"
        df["severity_category_lbl"] = df["severity_score"].apply(derive_cat)
    # Cast drivers to string to avoid mixed-type parquet save errors
    if "crisis_drivers" in df.columns:
        df["crisis_drivers"] = df["crisis_drivers"].astype(str)
    if "severity_category_lbl" in df.columns:
        df["severity_category_lbl"] = df["severity_category_lbl"].astype(str)
    return df, None

# ============================================================
# 3) Load every file into long format
# ============================================================
out(f"\nLoading {len(inform_files)} files...")
rows = []
skipped = []
t0 = time.time()

for i, finfo in enumerate(inform_files):
    df, err = read_country_sheet(finfo["path"])
    if df is None:
        skipped.append((finfo["name"], err))
        continue
    df = df.copy()
    df["year"]        = finfo["year"]
    df["month"]       = finfo["month"]
    df["source_file"] = finfo["name"]
    rows.append(df)
    if (i + 1) % 10 == 0 or i == len(inform_files) - 1:
        elapsed = time.time() - t0
        out(f"  {i+1:3d}/{len(inform_files)} files ({elapsed:.1f}s elapsed)")

if skipped:
    out(f"\nSkipped {len(skipped)} files:")
    for name, err in skipped:
        out(f"  - {name}: {err}")

inform_timeseries = pd.concat(rows, ignore_index=True)
# Belt-and-suspenders dedupe at row level in case any file had duplicate ISO3
inform_timeseries = inform_timeseries.drop_duplicates(
    subset=["iso3", "year", "month"], keep="first"
).reset_index(drop=True)
inform_timeseries["date_idx"] = inform_timeseries["year"] * 12 + inform_timeseries["month"]

out(f"\ninform_timeseries built: {len(inform_timeseries):,} rows")
out(f"  Unique countries: {inform_timeseries['iso3'].nunique()}")
out(f"  Unique year-months: {inform_timeseries[['year','month']].drop_duplicates().shape[0]}")

# ============================================================
# 4) Detect and report scale changes (median severity month-over-month)
# ============================================================
out(f"\n{'='*70}\nScale-change detection (median severity over time)\n{'='*70}")
monthly_median = (inform_timeseries
    .groupby(["year", "month"], as_index=False)["severity_score"]
    .median()
    .sort_values(["year", "month"])
    .reset_index(drop=True))
monthly_median["prev"] = monthly_median["severity_score"].shift(1)
monthly_median["ratio"] = monthly_median["severity_score"] / monthly_median["prev"]
scale_jumps = monthly_median[monthly_median["ratio"] > 1.5]
if len(scale_jumps) > 0:
    out("Month-over-month median severity jumps > 1.5x (likely scale changes):")
    for _, r in scale_jumps.iterrows():
        out(f"  {int(r['year'])}-{int(r['month']):02d}: median jumped from "
            f"{r['prev']:.2f} to {r['severity_score']:.2f}  (ratio {r['ratio']:.2f}x)")
    out("\nIMPLICATION: Raw numerical comparisons across these dates are unsafe.")
    out("This v2 uses CATEGORICAL labels (Very Low/Low/Medium/High/Very High)")
    out("which are scale-invariant by design.")
else:
    out("No month-over-month median jumps > 1.5x detected.")

# ============================================================
# 5) Per-country signals (CATEGORY-BASED, scale-invariant)
# ============================================================
ANALYSIS_END_DATE_IDX = inform_timeseries["date_idx"].max()

def country_signals(group):
    g = group.sort_values("date_idx").drop_duplicates(subset=["date_idx"], keep="first")
    n_obs = len(g)

    cutoff_24 = ANALYSIS_END_DATE_IDX - WINDOW_24 + 1
    cutoff_12 = ANALYSIS_END_DATE_IDX - WINDOW_12 + 1
    recent_24 = g[g["date_idx"] >= cutoff_24]
    recent_12 = g[g["date_idx"] >= cutoff_12]

    # Scale-invariant chronic-severity signals (using category labels)
    cats_24 = recent_24["severity_category_lbl"].fillna("")
    months_in_high_plus_24mo = cats_24.isin(HIGH_PLUS_CATEGORIES).sum()
    months_very_high_24mo    = cats_24.isin(VERY_HIGH_CATEGORY).sum()

    # Trend-based worsening signals (last 12 months)
    trends_12 = recent_12["severity_trend"].fillna("")
    months_increasing_12mo = (trends_12 == "Increasing").sum()
    months_decreasing_12mo = (trends_12 == "Decreasing").sum()

    # Latest observation
    if n_obs > 0:
        latest = g.iloc[-1]
        latest_severity_score    = latest["severity_score"]
        latest_severity_category = latest.get("severity_category_lbl")
        latest_trend             = latest.get("severity_trend")
        latest_month             = f"{int(latest['year'])}-{int(latest['month']):02d}"
        country_name             = latest.get("country_name")
    else:
        latest_severity_score = np.nan
        latest_severity_category = None
        latest_trend = None
        latest_month = None
        country_name = None

    # structural_neglect_v2: severity has been High or Very High for
    # >= 18 of the last 24 months (scale-invariant)
    structural_neglect_v2 = bool(months_in_high_plus_24mo >= 18)

    return pd.Series({
        "country_name":                 country_name,
        "n_observations":               n_obs,
        "n_observations_24mo":          len(recent_24),
        "latest_month":                 latest_month,
        "latest_severity_score":        latest_severity_score,
        "latest_severity_category":     latest_severity_category,
        "latest_trend":                 latest_trend,
        "months_in_high_plus_24mo":     int(months_in_high_plus_24mo),
        "months_very_high_24mo":        int(months_very_high_24mo),
        "months_increasing_12mo":       int(months_increasing_12mo),
        "months_decreasing_12mo":       int(months_decreasing_12mo),
        "structural_neglect_v2":        structural_neglect_v2,
    })

out(f"\nComputing per-country signals (categorical, scale-invariant)...")
inform_country_signals = (inform_timeseries
    .groupby("iso3", as_index=False)
    .apply(country_signals)
    .reset_index(drop=True))
out(f"  Signals computed for {len(inform_country_signals)} countries")

# ============================================================
# 6) Headline findings
# ============================================================
out(f"\n{'='*70}\nTop 20 by months_in_high_plus_24mo (chronic severity)\n{'='*70}")
chronic = inform_country_signals.sort_values(
    ["months_in_high_plus_24mo", "months_very_high_24mo"], ascending=False
).head(20)
disp_cols = ["iso3", "country_name", "n_observations_24mo",
             "months_in_high_plus_24mo", "months_very_high_24mo",
             "months_increasing_12mo", "months_decreasing_12mo",
             "latest_severity_category", "latest_trend",
             "structural_neglect_v2"]
out(chronic[disp_cols].to_string(index=False))

out(f"\n{'='*70}\nstructural_neglect_v2 fires for these countries:\n{'='*70}")
struct = inform_country_signals[inform_country_signals["structural_neglect_v2"]]
out(f"  {len(struct)} countries: severity has been High or Very High for ≥ 18 of last 24 months")
if len(struct) > 0:
    out(struct[["iso3", "country_name",
                "months_in_high_plus_24mo", "months_very_high_24mo",
                "latest_severity_category", "latest_trend"]].to_string(index=False))

out(f"\n{'='*70}\nTop 10 by months_increasing_12mo (worsening trend signal)\n{'='*70}")
worsening = inform_country_signals.sort_values(
    ["months_increasing_12mo", "months_in_high_plus_24mo"], ascending=False
).head(10)
out(worsening[["iso3", "country_name", "months_increasing_12mo",
               "months_decreasing_12mo", "months_in_high_plus_24mo",
               "latest_severity_category", "latest_trend"]].to_string(index=False))

# ============================================================
# 7) Trajectory verification — category labels over time
# ============================================================
out(f"\n{'='*70}\nTrajectory check (last 12 months by category, scale-invariant)\n{'='*70}")
for code in ["AFG", "SDN", "LBN", "PSE", "SYR", "BFA", "MMR"]:
    trace = (inform_timeseries[inform_timeseries["iso3"] == code]
             .sort_values("date_idx")
             .tail(12))
    if len(trace) == 0:
        out(f"\n{code}: no rows")
        continue
    out(f"\n{code}:")
    for _, row in trace.iterrows():
        out(f"  {int(row['year'])}-{int(row['month']):02d}  "
            f"severity {row['severity_score']:.1f}  "
            f"category={row.get('severity_category_lbl','-')!s:10s}  "
            f"trend={row.get('severity_trend','-')}")

# ============================================================
# 8) Save outputs
# ============================================================
try:
    inform_timeseries.to_parquet(OUTPUT_TS_PQT, index=False)
    inform_country_signals.to_parquet(OUTPUT_SIG_PQT, index=False)
    out(f"\nParquet save succeeded.")
except Exception as e:
    out(f"\nParquet save warning: {e} -- falling back to CSV")
    inform_timeseries.to_csv(OUTPUT_TS_PQT.replace(".parquet", ".csv"), index=False)
    inform_country_signals.to_csv(OUTPUT_SIG_PQT.replace(".parquet", ".csv"), index=False)

with open(OUTPUT_TXT, "w") as f:
    f.write("\n".join(_buf))

print(f"\n\n>>> Outputs written:")
print(f"    text log:         {OUTPUT_TXT}")
print(f"    timeseries pqt:   {OUTPUT_TS_PQT}")
print(f"    country signals:  {OUTPUT_SIG_PQT}")
print(f"\n>>> To view text output:")
print(f"    print(open('{OUTPUT_TXT}').read())")


INFORM SEVERITY TIME SERIES — bonus task (v2 — scale-invariant)

Discovered 63 candidate files; deduped to 62 (one per year-month)
  Duplicates dropped (1):
    - 202604-inform-severity-april-2026-1.xlsx
  Date range: 2021-01  ->  2026-04

Loading 62 files...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3

   10/62 files (12.6s elapsed)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3

   20/62 files (24.1s elapsed)


...

   30/62 files (36.0s elapsed)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3

   40/62 files (47.6s elapsed)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-bf99464b-9477-4ac3-b732-c41bf469a1f9/lib/python3.10/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'INFORM Severity - all crises'!$1:$67.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


   50/62 files (60.1s elapsed)
   60/62 files (70.8s elapsed)
   62/62 files (72.5s elapsed)

inform_timeseries built: 4,927 rows
  Unique countries: 108
  Unique year-months: 62

Scale-change detection (median severity over time)
Month-over-month median severity jumps > 1.5x (likely scale changes):
  2026-01: median jumped from 3.10 to 6.15  (ratio 1.98x)

IMPLICATION: Raw numerical comparisons across these dates are unsafe.
This v2 uses CATEGORICAL labels (Very Low/Low/Medium/High/Very High)
which are scale-invariant by design.

Computing per-country signals (categorical, scale-invariant)...
  Signals computed for 108 countries

Top 20 by months_in_high_plus_24mo (chronic severity)
iso3 country_name  n_observations_24mo  months_in_high_plus_24mo  months_very_high_24mo  months_increasing_12mo  months_decreasing_12mo latest_severity_category latest_trend  structural_neglect_v2
 AFG  Afghanistan                   22                        22                     22                       

In [0]:
print(open('/tmp/inform_timeseries_output.txt').read())

INFORM SEVERITY TIME SERIES — bonus task (v2 — scale-invariant)

Discovered 63 candidate files; deduped to 62 (one per year-month)
  Duplicates dropped (1):
    - 202604-inform-severity-april-2026-1.xlsx
  Date range: 2021-01  ->  2026-04

Loading 62 files...
   10/62 files (12.6s elapsed)
   20/62 files (24.1s elapsed)
   30/62 files (36.0s elapsed)
   40/62 files (47.6s elapsed)
   50/62 files (60.1s elapsed)
   60/62 files (70.8s elapsed)
   62/62 files (72.5s elapsed)

inform_timeseries built: 4,927 rows
  Unique countries: 108
  Unique year-months: 62

Scale-change detection (median severity over time)
Month-over-month median severity jumps > 1.5x (likely scale changes):
  2026-01: median jumped from 3.10 to 6.15  (ratio 1.98x)

IMPLICATION: Raw numerical comparisons across these dates are unsafe.
This v2 uses CATEGORICAL labels (Very Low/Low/Medium/High/Very High)
which are scale-invariant by design.

Computing per-country signals (categorical, scale-invariant)...
  Signals compu

In [0]:
# ============================================================
# Merge INFORM temporal signals into the gap-score outputs.
#
# Reads inform_country_signals (from inform_timeseries_v1.py) and joins it
# into ranked_gap_scores and country_rollup on iso3. Replaces the v1
# structural_neglect_flag with the working v2 signal.
#
# Run this AFTER both gap_score_v1.py and inform_timeseries_v1.py have completed.
# Expects in scope: ranked_gap_scores, country_rollup, inform_country_signals
#                   (or reads the parquet files from /tmp/ as fallback).
# ============================================================
import os
import pandas as pd

# ============================================================
# Load inform_country_signals (from memory if available, else from disk)
# ============================================================
try:
    signals = inform_country_signals
    print(f"Using inform_country_signals from memory: {signals.shape}")
except NameError:
    path = "/tmp/inform_country_signals.parquet"
    if os.path.exists(path):
        signals = pd.read_parquet(path)
    else:
        path_csv = "/tmp/inform_country_signals.csv"
        if os.path.exists(path_csv):
            signals = pd.read_csv(path_csv)
        else:
            raise RuntimeError("inform_country_signals not found in memory or /tmp/. "
                              "Run inform_timeseries_v1.py first.")
    print(f"Loaded inform_country_signals from {path}: {signals.shape}")

# Columns to merge in
temporal_cols = [
    "iso3",
    "months_in_high_plus_24mo",
    "months_very_high_24mo",
    "months_increasing_12mo",
    "months_decreasing_12mo",
    "structural_neglect_v2",
]
signals_for_merge = signals[temporal_cols].copy()

# ============================================================
# Merge into ranked_gap_scores
# ============================================================
# Drop the broken v1 structural_neglect_flag if present (we'll replace it)
if "structural_neglect_flag" in ranked_gap_scores.columns:
    ranked_gap_scores = ranked_gap_scores.drop(columns=["structural_neglect_flag"])

ranked_gap_scores = ranked_gap_scores.merge(signals_for_merge, on="iso3", how="left")
# Rename for compatibility — v2 is the canonical flag now
ranked_gap_scores = ranked_gap_scores.rename(
    columns={"structural_neglect_v2": "structural_neglect_flag"}
)
# Fill NaN with False for countries with no INFORM time-series data
ranked_gap_scores["structural_neglect_flag"] = (
    ranked_gap_scores["structural_neglect_flag"].fillna(False).astype(bool)
)
print(f"\nranked_gap_scores updated: {ranked_gap_scores.shape}")
print(f"  Now has temporal signals: months_in_high_plus_24mo, months_very_high_24mo, "
      f"months_increasing_12mo, months_decreasing_12mo")
print(f"  structural_neglect_flag (v2) fires for "
      f"{ranked_gap_scores['structural_neglect_flag'].sum()} plan-rows")

# ============================================================
# Merge into country_rollup too
# ============================================================
if "any_structural_neglect" in country_rollup.columns:
    country_rollup = country_rollup.drop(columns=["any_structural_neglect"])

country_rollup = country_rollup.merge(signals_for_merge, on="iso3", how="left")
country_rollup = country_rollup.rename(
    columns={"structural_neglect_v2": "any_structural_neglect"}
)
country_rollup["any_structural_neglect"] = (
    country_rollup["any_structural_neglect"].fillna(False).astype(bool)
)
print(f"\ncountry_rollup updated: {country_rollup.shape}")
print(f"  any_structural_neglect fires for {country_rollup['any_structural_neglect'].sum()} country-year rows")

# ============================================================
# Print the new view: Tier 1 plans + structural_neglect_v2 status
# ============================================================
print(f"\n{'='*80}")
print("Tier 1 plans + temporal signals")
print(f"{'='*80}")
tier1 = ranked_gap_scores[ranked_gap_scores["gap_tier"] == "Tier 1 (most overlooked)"]
cols_show = ["iso3", "plan_code", "plan_type", "gap_score",
             "months_in_high_plus_24mo", "months_very_high_24mo",
             "months_increasing_12mo", "structural_neglect_flag"]
print(tier1[cols_show].to_string(index=False))

# ============================================================
# Cross-reference: which structural_neglect_v2 countries are NOT in our crises table?
# ============================================================
print(f"\n{'='*80}")
print("Chronic-neglect countries MISSING from our gap-score ranking")
print(f"(structural_neglect_v2 fires, but no FTS plan in 2024-2026)")
print(f"{'='*80}")
chronic_countries = set(signals[signals["structural_neglect_v2"]]["iso3"])
ranked_countries  = set(ranked_gap_scores["iso3"])
missing = chronic_countries - ranked_countries
if missing:
    missing_signals = signals[signals["iso3"].isin(missing)].sort_values(
        ["months_very_high_24mo", "months_in_high_plus_24mo"], ascending=False
    )
    cols = ["iso3", "country_name", "months_in_high_plus_24mo",
            "months_very_high_24mo", "latest_severity_category", "latest_trend"]
    print(missing_signals[cols].to_string(index=False))
    print(f"\n  -> {len(missing)} chronic-neglect countries are invisible to our funding-gap ranking.")
    print(f"     They have no formal plan in FTS 2024-2026. (Some appear in documented_need_no_plan.)")
else:
    print("  (none — every structural_neglect_v2 country has a plan in our ranking)")

# ============================================================
# Top rising-risk countries (high months_increasing, not yet Tier 1/2)
# ============================================================
print(f"\n{'='*80}")
print("Rising-risk: countries with worsening 12-month trend (top 10 by months_increasing_12mo)")
print(f"{'='*80}")
rising = signals.sort_values(
    ["months_increasing_12mo", "months_in_high_plus_24mo"], ascending=False
).head(10)
rising_cols = ["iso3", "country_name", "months_increasing_12mo",
               "months_in_high_plus_24mo", "latest_severity_category", "latest_trend"]
print(rising[rising_cols].to_string(index=False))

# ============================================================
# CRITICAL: persist the updated dataframes back to /tmp/ so save_outputs.py
# picks up the v2 structural_neglect_flag and the temporal-signal columns.
# Without this, the parquets on disk still have v1 (gap_score_v1.py output)
# and downstream consumers (dashboard, Genie, fresh notebook runs) see stale
# v1 data even though the in-memory dataframes are correct.
# ============================================================
import os
RANKED_PARQUET = "/tmp/gap_score_v1_ranked.parquet"
ROLLUP_PARQUET = "/tmp/gap_score_v1_country_rollup.parquet"

try:
    ranked_gap_scores.to_parquet(RANKED_PARQUET, index=False)
    country_rollup.to_parquet(ROLLUP_PARQUET, index=False)
    print(f"\nPersisted updated dataframes back to /tmp/:")
    print(f"  {RANKED_PARQUET}  ({len(ranked_gap_scores)} rows x {ranked_gap_scores.shape[1]} cols)")
    print(f"  {ROLLUP_PARQUET}  ({len(country_rollup)} rows x {country_rollup.shape[1]} cols)")
    print(f"  -> run save_outputs.py next to push to durable workspace storage.")
except Exception as e:
    print(f"\nWARNING: failed to persist updated dataframes: {e}")
    print(f"  Falling back to CSV...")
    ranked_gap_scores.to_csv(RANKED_PARQUET.replace(".parquet", ".csv"), index=False)
    country_rollup.to_csv(ROLLUP_PARQUET.replace(".parquet", ".csv"), index=False)


Using inform_country_signals from memory: (108, 13)

ranked_gap_scores updated: (239, 40)
  Now has temporal signals: months_in_high_plus_24mo, months_very_high_24mo, months_increasing_12mo, months_decreasing_12mo
  structural_neglect_flag (v2) fires for 137 plan-rows

country_rollup updated: (199, 27)
  any_structural_neglect fires for 102 country-year rows

Tier 1 plans + temporal signals
iso3 plan_code                            plan_type  gap_score  months_in_high_plus_24mo  months_very_high_24mo  months_increasing_12mo  structural_neglect_flag
 LBN    RSYR25               Regional response plan   0.608909                      22.0                    5.0                     1.0                     True
 COD    HCOD25 Humanitarian needs and response plan   0.601746                      22.0                   22.0                     3.0                     True
 YEM    HYEM25 Humanitarian needs and response plan   0.595304                      22.0                   22.0            

## Stage 5 — Geospatial deliverable (the PDF's "map-ready output")

### What this stage does

Renders the geospatial deliverable the PDF asks for under expected outputs:

> *"map-ready outputs using country or crisis coordinates where available"*

A Plotly `scatter_geo` world map with three layers:

| Layer | Render | What it shows |
|---|---|---|
| Tier 1 (most overlooked) | Deep red filled circles | 4 plans, marker size = absolute funding gap |
| Tier 2 (severely overlooked) | Orange filled circles | 22 plans, marker size = absolute funding gap |
| Documented need, no plan | Open blue circles | 8 countries (Mauritania + 7 others) at INFORM severity ≥ 5 |

### What the map shows visually

- **The eye lands first on Palestine FPSE26** — the largest marker on the entire map ($3.58B gap), but **orange** (Tier 2), not red. This is the multi-lens paradox §6.6 of the writeup defends: largest absolute gap, but not the most overlooked by score because Palestine is 12% raw / ~32% annualized funded.
- **Lebanon RSYR25 is the largest red marker** ($2.72B Tier 1 gap) — the Syria 3RP headline finding visually leads the eye among the Tier 1 set.
- **The chronic-crisis belt is geographically visible** — Tier 2 markers cluster across the Sahel (Mali, Burkina Faso, Chad, Sudan), the Horn (Ethiopia, Somalia), MENA (Syria, Yemen, Palestine, Lebanon, Türkiye), Latin America (Haiti, Colombia, Honduras, Venezuela), and Asia (Afghanistan, Myanmar).
- **The open blue circles** answer the PDF's design challenge "*countries with active need but no formal Humanitarian Response Plan in place*" — Mauritania in the Sahel, Eritrea in the Horn, Algeria + Morocco in North Africa, Indonesia + Thailand in SE Asia, Namibia in southern Africa, Jamaica in the Caribbean.

### Output of this stage

- `gap_map.html` — interactive Plotly map (hover reveals plan code, type, coverage %, gap_score, op_env, structural_neglect flag)
- `gap_map.png` — static 1400×800 export for inclusion in a brief (requires `kaleido`; the cell falls back gracefully if unavailable)

Both saved directly to `/Workspace/Users/<your-email>/asli_gulcur_submission/` — no separate save step required.


In [0]:
# ============================================================
# GAP MAP — static world map of Tier 1 / Tier 2 plans
#
# Renders a Plotly scatter_geo:
#   marker size  = absolute funding gap (sum_funding_gap_usd, country_rollup)
#   marker color = gap_tier
#   hover text   = country, plan(s), coverage %, gap_score, op_env
#
# Saves HTML + PNG to durable workspace storage so an assessor can
# open the map without re-running the pipeline.
#
# Requires in scope: ranked_gap_scores, country_rollup
# Run after: gap_score_v1.py + merge_temporal_signals.py
# ============================================================
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ============================================================
# 1) Build the per-plan view (one marker per Tier 1/2 plan)
# ============================================================
plans = ranked_gap_scores[
    ranked_gap_scores["gap_tier"].isin([
        "Tier 1 (most overlooked)",
        "Tier 2 (severely overlooked)",
    ])
].copy()

# Absolute funding gap per plan (raw USD)
plans["funding_gap_usd"] = (
    plans["fts_requirements"].fillna(0) - plans["fts_funding"].fillna(0)
).clip(lower=0)
plans["funding_gap_billions"] = plans["funding_gap_usd"] / 1e9

# Drop rows without coordinates (can't plot them)
plans = plans.dropna(subset=["latitude", "longitude"])

# Build a compact hover label
def hover(row):
    op_env = (f"op_env {row['operating_env_score']:.1f}"
              if pd.notna(row.get("operating_env_score"))
              else "op_env n/a")
    cov = (f"{row['fts_pct_funded']:.0f}%"
           if pd.notna(row.get("fts_pct_funded"))
           else "n/a")
    sn = " | structural_neglect" if row.get("structural_neglect_flag") else ""
    return (
        f"<b>{row['country_name']}</b><br>"
        f"{row['plan_code']} ({row['plan_type']}) {int(row['year'])}<br>"
        f"Coverage: {cov} &nbsp;|&nbsp; gap_score: {row['gap_score']:.2f}<br>"
        f"Gap: ${row['funding_gap_billions']:.2f}B &nbsp;|&nbsp; {op_env}{sn}<br>"
        f"<i>{row['gap_tier']}</i>"
    )

plans["hover_text"] = plans.apply(hover, axis=1)

# Jitter overlapping markers so a country with multiple plans (e.g. Yemen's
# HYEM25 Tier 1 + HYEM26 Tier 2, or Afghanistan's HAFG25 + HAFG26) doesn't
# render as one dot. Sort by gap_score so the most-overlooked plan stays on top.
plans = plans.sort_values("gap_score", ascending=False).reset_index(drop=True)
plans["_offset_idx"] = plans.groupby("iso3").cumcount()
JITTER_DEG = 1.8  # ~200km separation between stacked markers
plans["latitude"]  = plans["latitude"]  + plans["_offset_idx"] * JITTER_DEG
plans["longitude"] = plans["longitude"] + plans["_offset_idx"] * JITTER_DEG

# ============================================================
# 2) Tier colors — use a humanitarian-coherent palette
# ============================================================
TIER_COLOR = {
    "Tier 1 (most overlooked)":     "#b30000",  # deep red — urgent
    "Tier 2 (severely overlooked)": "#fc8d59",  # orange — high concern
}

# ============================================================
# 3) Build the Plotly figure
# ============================================================
fig = go.Figure()

for tier_label, color in TIER_COLOR.items():
    sub = plans[plans["gap_tier"] == tier_label]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Scattergeo(
        lon=sub["longitude"],
        lat=sub["latitude"],
        text=sub["hover_text"],
        hoverinfo="text",
        name=tier_label,
        marker=dict(
            size=sub["funding_gap_billions"].clip(lower=0.05) * 12 + 8,
            color=color,
            line=dict(width=1, color="black"),
            opacity=0.85,
            sizemode="diameter",
        ),
    ))

# Documented-need-no-plan countries as a tertiary layer (open circles)
# Backfill centroids for countries the main CENTROIDS dict in gap_score_v1.py
# does not include — these 8 countries surfaced in documented_need_no_plan
# (Mauritania, Eritrea, Thailand, Algeria, Indonesia, Namibia, Morocco, Jamaica)
# all lack centroids upstream, which would otherwise silently drop them.
NO_PLAN_CENTROID_FALLBACK = {
    "MRT": (21.008, -10.941),   # Mauritania
    "ERI": (15.179,  39.782),   # Eritrea
    "THA": (15.870, 100.992),   # Thailand
    "DZA": (28.034,   1.659),   # Algeria
    "IDN": (-0.789, 113.921),   # Indonesia
    "NAM": (-22.957, 18.490),   # Namibia
    "MAR": (31.792,  -7.092),   # Morocco
    "JAM": (18.109, -77.297),   # Jamaica
}
try:
    no_plan = documented_need_no_plan.copy()
    no_plan["latitude"]  = no_plan.apply(
        lambda r: r["latitude"]  if pd.notna(r["latitude"])
                  else NO_PLAN_CENTROID_FALLBACK.get(r["iso3"], (None, None))[0],
        axis=1,
    )
    no_plan["longitude"] = no_plan.apply(
        lambda r: r["longitude"] if pd.notna(r["longitude"])
                  else NO_PLAN_CENTROID_FALLBACK.get(r["iso3"], (None, None))[1],
        axis=1,
    )
    no_plan = no_plan.dropna(subset=["latitude", "longitude"])
    if len(no_plan) > 0:
        no_plan["hover_text"] = no_plan.apply(
            lambda r: (
                f"<b>{r['country_name']}</b><br>"
                f"INFORM severity {r['severity_score']:.1f} ({r.get('severity_category_lbl','-')})<br>"
                f"Trend: {r.get('severity_trend','-')}<br>"
                f"<i>Documented need, no formal plan</i>"
            ),
            axis=1,
        )
        fig.add_trace(go.Scattergeo(
            lon=no_plan["longitude"],
            lat=no_plan["latitude"],
            text=no_plan["hover_text"],
            hoverinfo="text",
            name="Documented need, no plan",
            marker=dict(
                size=12,
                color="rgba(0,0,0,0)",
                line=dict(width=2, color="#0570b0"),
                symbol="circle",
            ),
        ))
except NameError:
    pass  # documented_need_no_plan not in scope; skip layer

fig.update_layout(
    title=dict(
        text=("Geo-Insight: Most-overlooked humanitarian plans (Tier 1 + Tier 2)<br>"
              "<sub>Marker size = absolute funding gap (USD billions); "
              "color = gap tier; open circles = documented need with no formal plan</sub>"),
        x=0.5,
    ),
    geo=dict(
        showland=True,
        landcolor="#f4f4f4",
        showcountries=True,
        countrycolor="#cccccc",
        showocean=True,
        oceancolor="#eaf2f8",
        projection_type="natural earth",
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.05,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(l=10, r=10, t=80, b=40),
    height=700,
)

# Show inline in the notebook
fig.show()

# ============================================================
# 4) Save HTML + (optional) PNG to durable storage
# ============================================================
try:
    current_user = spark.sql("SELECT current_user()").collect()[0][0]
except Exception:
    current_user = "unknown_user"

out_dir = f"/Workspace/Users/{current_user}/asli_gulcur_submission"
os.makedirs(out_dir, exist_ok=True)

html_path = f"{out_dir}/gap_map.html"
fig.write_html(html_path, include_plotlyjs="cdn")
print(f"\nMap saved: {html_path}")

# PNG requires kaleido — try, but don't fail the cell if unavailable
try:
    png_path = f"{out_dir}/gap_map.png"
    fig.write_image(png_path, width=1400, height=800, scale=2)
    print(f"Map PNG saved: {png_path}")
except Exception as e:
    print(f"PNG export skipped ({e}). HTML is the canonical deliverable.")
    print("To enable PNG: %pip install kaleido && dbutils.library.restartPython()")

# ============================================================
# 5) Print a brief summary so the cell output isn't silent
# ============================================================
print(f"\nPlotted {len(plans)} plan markers across "
      f"{plans['iso3'].nunique()} countries:")
for tier in ("Tier 1 (most overlooked)", "Tier 2 (severely overlooked)"):
    sub = plans[plans["gap_tier"] == tier]
    if len(sub) > 0:
        print(f"  {tier}: {len(sub)} plans, "
              f"total gap ${sub['funding_gap_billions'].sum():.2f}B")
try:
    print(f"  Documented need, no plan: {len(no_plan)} countries")
except NameError:
    pass



Map saved: /Workspace/Users/agulcur@andrew.cmu.edu/asli_gulcur_submission/gap_map.html
PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
). HTML is the canonical deliverable.
To enable PNG: %pip install kaleido && dbutils.library.restartPython()

Plotted 26 plan markers across 19 countries:
  Tier 1 (most overlooked): 4 plans, total gap $7.27B
  Tier 2 (severely overlooked): 22 plans, total gap $26.57B
  Documented need, no plan: 8 countries


## Stage 6 — Durable storage

### Why this stage exists

The previous stages write output files to `/tmp/`, which is cluster-local. When the cluster shuts down (typical for Serverless after idle), the outputs vanish — and any downstream consumer (an assessor running this on a fresh cluster, a follow-up dashboard, the verification_test) can't read them.

This stage copies the outputs to a workspace path that survives cluster restarts. The path is auto-detected from `current_user()` so this notebook works for any user who runs it — no edits needed.

### Output path

Outputs land at: `/Workspace/Users/<your-email>/asli_gulcur_submission/`

Eleven files total (10 copied by this stage; `gap_map.html` was already written directly in Stage 5):

**Stage 3 — Gap score outputs:**
- `gap_score_v1_output.txt` — human-readable summary log
- `gap_score_v1_ranked.parquet` — `ranked_gap_scores` (239 × 40)
- `gap_score_v1_country_rollup.parquet` — `country_rollup` (199 × 27)
- `gap_score_v1_no_plan.parquet` — `documented_need_no_plan` (8 rows)
- `gap_score_v1_sensitivity.json` — sensitivity test results, including the new `tier1_set_membership_rates` per scenario

**Stage 4 — Bonus task outputs:**
- `inform_timeseries_output.txt` — bonus-task log with scale-change detection + trajectory traces
- `inform_timeseries.parquet` — 4,927 × 9 long-format time series
- `inform_country_signals.parquet` — 108 × 13 per-country aggregate signals
- (CSV fallbacks for both, in case Parquet write fails on the runtime)

**Stage 5 — Geospatial deliverable:**
- `gap_map.html` — interactive map (written directly by Stage 5)
- `gap_map.png` — static PNG, if `kaleido` is available


In [0]:
# ============================================================
# Save outputs to durable storage so the command center can read them
# even after cluster restart.
#
# On Databricks Serverless:
#   - /Volumes/cmu_hackathon/common/unocha/ is read-only (shared challenge volume)
#   - /dbfs/ FUSE mount is not available
#   - dbutils.fs works for DBFS paths
#   - /Workspace/Users/<email>/ is writable for the current user
#
# This script tries paths in priority order and reports what worked.
# ============================================================
import os
import shutil

# Auto-detect the current Databricks user
try:
    current_user = spark.sql("SELECT current_user()").collect()[0][0]
except Exception:
    current_user = "unknown_user"

src_files = [
    # Stage 3 gap-score outputs
    "/tmp/gap_score_v1_output.txt",
    "/tmp/gap_score_v1_ranked.parquet",
    "/tmp/gap_score_v1_country_rollup.parquet",
    "/tmp/gap_score_v1_no_plan.parquet",
    "/tmp/gap_score_v1_sensitivity.json",
    # Stage 5 INFORM time series outputs (bonus task)
    "/tmp/inform_timeseries_output.txt",
    "/tmp/inform_timeseries.parquet",
    "/tmp/inform_country_signals.parquet",
    # Or CSV fallbacks if parquet failed
    "/tmp/inform_timeseries.csv",
    "/tmp/inform_country_signals.csv",
]

# Build candidate destinations in priority order
candidates = [
    ("workspace",   f"/Workspace/Users/{current_user}/asli_gulcur_submission",   "filesystem"),
    ("volume_user", f"/Volumes/cmu_hackathon/common/unocha/{current_user}",      "filesystem"),
    ("dbfs",        f"dbfs:/FileStore/{current_user}/asli_gulcur_submission",    "dbutils"),
]

print(f"Current Databricks user: {current_user}\n")

def try_filesystem(dest_dir):
    try:
        os.makedirs(dest_dir, exist_ok=True)
        for f in src_files:
            if os.path.exists(f):
                shutil.copy(f, dest_dir)
        return True
    except (PermissionError, OSError) as e:
        return False, str(e)

def try_dbutils(dest_dir):
    try:
        dbutils.fs.mkdirs(dest_dir)
        for f in src_files:
            if os.path.exists(f):
                dbutils.fs.cp(f"file:{f}", f"{dest_dir}/{os.path.basename(f)}")
        return True
    except Exception as e:
        return False, str(e)

final_dir = None
for label, path, method in candidates:
    print(f"Trying ({label}) {path} via {method}...")
    if method == "filesystem":
        result = try_filesystem(path)
    else:
        result = try_dbutils(path)
    if result is True:
        final_dir = path
        method_used = method
        print(f"  SUCCESS")
        break
    else:
        print(f"  FAILED: {result[1] if isinstance(result, tuple) else result}")

if final_dir:
    print(f"\n>>> Outputs durably saved to: {final_dir}")
    print(f">>> Contents:")
    if method_used == "filesystem":
        for f in sorted(os.listdir(final_dir)):
            size_kb = os.path.getsize(f"{final_dir}/{f}") / 1024
            print(f"    {size_kb:8.1f} KB  {f}")
    else:
        for entry in sorted(dbutils.fs.ls(final_dir), key=lambda x: x.name):
            size_kb = entry.size / 1024
            print(f"    {size_kb:8.1f} KB  {entry.name}")
    print(f"\n>>> Path to use for command center (Notebook 2): {final_dir}")
else:
    print(f"\nERROR: no writable path found. Check workspace permissions.")


Current Databricks user: agulcur@andrew.cmu.edu

Trying (workspace) /Workspace/Users/agulcur@andrew.cmu.edu/asli_gulcur_submission via filesystem...
  SUCCESS

>>> Outputs durably saved to: /Workspace/Users/agulcur@andrew.cmu.edu/asli_gulcur_submission
>>> Contents:
        15.9 KB  gap_map.html
        34.3 KB  gap_score_v1_country_rollup.parquet
        13.3 KB  gap_score_v1_no_plan.parquet
        11.7 KB  gap_score_v1_output.txt
        59.0 KB  gap_score_v1_ranked.parquet
         1.0 KB  gap_score_v1_sensitivity.json
         9.7 KB  inform_country_signals.csv
        12.6 KB  inform_country_signals.parquet
       559.8 KB  inform_timeseries.csv
        27.5 KB  inform_timeseries.parquet
        16.5 KB  inform_timeseries_output.txt

>>> Path to use for command center (Notebook 2): /Workspace/Users/agulcur@andrew.cmu.edu/asli_gulcur_submission


In [0]:
# ============================================================
# VERIFICATION TEST — does the pipeline output match the writeup claims?
# ============================================================
# Run this AFTER all the pipeline cells have completed, INCLUDING
# inform_timeseries_v1.py and merge_temporal_signals.py.
#
# Expects in scope: crises, inform_country, ranked_gap_scores,
#                   country_rollup, documented_need_no_plan,
#                   inform_timeseries, inform_country_signals
#
# Prints a pass/fail report for every documented finding.
# ============================================================
import os
import pandas as pd

results = []  # list of (status, label, expected, actual, note)

def check(label, condition, expected, actual, note=""):
    status = "PASS" if condition else "FAIL"
    results.append((status, label, str(expected), str(actual), note))

# ============================================================
# 1) DataFrame shapes
# ============================================================
check("crises rows",
      crises.shape[0] == 239,
      239, crises.shape[0])

check("crises columns",
      crises.shape[1] == 14,
      14, crises.shape[1])

check("inform_country rows",
      inform_country.shape[0] == 68,
      68, inform_country.shape[0])

check("ranked_gap_scores rows",
      ranked_gap_scores.shape[0] == 239,
      239, ranked_gap_scores.shape[0])

check("country_rollup rows",
      country_rollup.shape[0] == 199,
      199, country_rollup.shape[0])

check("documented_need_no_plan rows",
      documented_need_no_plan.shape[0] == 8,
      8, documented_need_no_plan.shape[0])

# ============================================================
# 2) Tier distribution
# ============================================================
tier_counts = ranked_gap_scores["gap_tier"].value_counts().to_dict()

check("Tier 1 count = 4",
      tier_counts.get("Tier 1 (most overlooked)") == 4,
      4, tier_counts.get("Tier 1 (most overlooked)"))

check("Tier 2 count = 22",
      tier_counts.get("Tier 2 (severely overlooked)") == 22,
      22, tier_counts.get("Tier 2 (severely overlooked)"))

check("Tier 3 count = 46",
      tier_counts.get("Tier 3 (underfunded)") == 46,
      46, tier_counts.get("Tier 3 (underfunded)"))

check("Tier 4 count = 167",
      tier_counts.get("Tier 4 (adequately resourced)") == 167,
      167, tier_counts.get("Tier 4 (adequately resourced)"))

# ============================================================
# 3) Tier 1 plans — the headline
# ============================================================
tier1 = ranked_gap_scores[
    ranked_gap_scores["gap_tier"] == "Tier 1 (most overlooked)"
].sort_values("gap_score", ascending=False)
tier1_codes = set(tier1["plan_code"].tolist())

check("Tier 1 = exactly 4 plans",
      len(tier1) == 4,
      4, len(tier1))

check("LBN RSYR25 (Syria 3RP) in Tier 1",
      "RSYR25" in tier1_codes,
      "in Tier 1", "found" if "RSYR25" in tier1_codes else "NOT FOUND")

check("COD HCOD25 (DRC) in Tier 1",
      "HCOD25" in tier1_codes,
      "in Tier 1", "found" if "HCOD25" in tier1_codes else "NOT FOUND")

check("YEM HYEM25 (Yemen) in Tier 1",
      "HYEM25" in tier1_codes,
      "in Tier 1", "found" if "HYEM25" in tier1_codes else "NOT FOUND")

check("MMR HMMR25b (Myanmar) in Tier 1",
      "HMMR25b" in tier1_codes,
      "in Tier 1", "found" if "HMMR25b" in tier1_codes else "NOT FOUND")

# Top plan should be Lebanon-RSYR25
top1 = tier1.iloc[0]
check("#1 plan is LBN RSYR25",
      top1["plan_code"] == "RSYR25" and top1["iso3"] == "LBN",
      "LBN RSYR25", f"{top1['iso3']} {top1['plan_code']}")

# ============================================================
# 4) Sudan paradox — severity 9.6 but Tier 2 (not Tier 1)
# ============================================================
sdn25 = ranked_gap_scores[
    (ranked_gap_scores["iso3"] == "SDN") &
    (ranked_gap_scores["plan_code"] == "HSDN25")
]
if len(sdn25) == 1:
    sdn_row = sdn25.iloc[0]
    check("Sudan HSDN25 severity = 9.6 (highest in dataset)",
          abs(sdn_row["severity_norm"] - 0.96) < 0.01,
          0.96, round(sdn_row["severity_norm"], 3))
    check("Sudan HSDN25 ranks Tier 2 (not Tier 1)",
          sdn_row["gap_tier"] == "Tier 2 (severely overlooked)",
          "Tier 2 (severely overlooked)", sdn_row["gap_tier"])
else:
    check("Sudan HSDN25 present", False, 1, len(sdn25), "row missing")

# ============================================================
# 5) Palestine: largest absolute gap + op_env = 10.0
# ============================================================
pse_rollup = country_rollup[country_rollup["iso3"] == "PSE"]
if len(pse_rollup) > 0:
    pse_2026 = pse_rollup[pse_rollup["year"] == 2026]
    if len(pse_2026) == 1:
        pse_row = pse_2026.iloc[0]
        # ~3.58B raw funding gap
        check("Palestine 2026 absolute gap ≈ $3.58B",
              abs(pse_row["sum_funding_gap_usd"] - 3.58e9) < 0.05e9,
              "≈ 3.58e9", f"{pse_row['sum_funding_gap_usd']:.2e}")
        check("Palestine 2026 op_env = 10.0",
              pse_row["operating_env_score"] == 10.0,
              10.0, pse_row["operating_env_score"])

# Palestine #1 by absolute gap globally
top_by_gap = country_rollup.sort_values("sum_funding_gap_usd", ascending=False).iloc[0]
check("Palestine 2026 is #1 by absolute funding gap globally",
      top_by_gap["iso3"] == "PSE" and top_by_gap["year"] == 2026,
      "PSE 2026", f"{top_by_gap['iso3']} {top_by_gap['year']}")

# ============================================================
# 6) Palestine has no HNO in any year (need_source = FTS_proxy)
# ============================================================
pse_plans = ranked_gap_scores[ranked_gap_scores["iso3"] == "PSE"]
if len(pse_plans) > 0:
    pse_need_sources = set(pse_plans["need_source"].dropna().unique())
    check("Palestine uses FTS_requirements_proxy (no HNO PIN)",
          pse_need_sources == {"FTS_requirements_proxy"},
          "{'FTS_requirements_proxy'}", str(pse_need_sources))

# ============================================================
# 7) Mauritania in documented_need_no_plan
# ============================================================
mrt = documented_need_no_plan[documented_need_no_plan["iso3"] == "MRT"]
check("Mauritania in documented_need_no_plan",
      len(mrt) == 1,
      1, len(mrt))

if len(mrt) == 1:
    mrt_row = mrt.iloc[0]
    check("Mauritania severity ≈ 6.1",
          abs(mrt_row["severity_score"] - 6.1) < 0.1,
          6.1, round(mrt_row["severity_score"], 2))
    check("Mauritania trend = Increasing",
          mrt_row["severity_trend"] == "Increasing",
          "Increasing", mrt_row["severity_trend"])

# ============================================================
# 8) Afghanistan trajectory (52% -> 42% -> 15%)
# ============================================================
for year, exp_pct, label in [(2024, 52, "AFG 2024 ≈ 52%"),
                              (2025, 42, "AFG 2025 ≈ 42%"),
                              (2026, 15, "AFG 2026 ≈ 15%")]:
    afg = ranked_gap_scores[
        (ranked_gap_scores["iso3"] == "AFG") &
        (ranked_gap_scores["plan_code"].str.startswith("HAFG")) &
        (ranked_gap_scores["year"] == year)
    ]
    if len(afg) == 1:
        actual_pct = afg.iloc[0]["fts_pct_funded"]
        check(label,
              abs(actual_pct - exp_pct) < 2,
              f"≈ {exp_pct}%", f"{actual_pct:.0f}%")

# ============================================================
# 9) Hidden sector gaps fire for SDN and ETH HRPs
# ============================================================
hidden_gaps = ranked_gap_scores[ranked_gap_scores["hidden_sector_gap"] == True]
hidden_codes = set(hidden_gaps["plan_code"].tolist())

check("hidden_sector_gap fires for SDN HSDN25",
      "HSDN25" in hidden_codes,
      "fires", "fires" if "HSDN25" in hidden_codes else "DOES NOT FIRE")
check("hidden_sector_gap fires for ETH HETH24",
      "HETH24" in hidden_codes,
      "fires", "fires" if "HETH24" in hidden_codes else "DOES NOT FIRE")

# ============================================================
# 10) Bonus task — structural_neglect_v2 fires for 37 countries
# ============================================================
try:
    _signals_available = True
    _ = inform_country_signals
except NameError:
    _signals_available = False
    check("inform_country_signals in scope",
          False, "available",
          "MISSING — run inform_timeseries_v1.py first")

if _signals_available:
    n_struct = int(inform_country_signals["structural_neglect_v2"].sum())
    check("structural_neglect_v2 fires for 37 countries",
          n_struct == 37,
          37, n_struct)

    # 11) 36 of 37 are in our gap-score ranking; only DPRK invisible
    chronic = set(
        inform_country_signals[inform_country_signals["structural_neglect_v2"]]["iso3"]
    )
    ranked_countries = set(ranked_gap_scores["iso3"])
    missing = chronic - ranked_countries
    check("36 of 37 chronic-neglect countries in ranking",
          len(chronic & ranked_countries) == 36,
          36, len(chronic & ranked_countries))
    check("Only DPRK (PRK) is the missing chronic-neglect country",
          missing == {"PRK"},
          "{'PRK'}", str(missing))

    # 12) Angola: 10 of last 12 months Increasing
    ago = inform_country_signals[inform_country_signals["iso3"] == "AGO"]
    if len(ago) == 1:
        ago_inc = int(ago.iloc[0]["months_increasing_12mo"])
        check("Angola months_increasing_12mo == 10",
              ago_inc == 10,
              10, ago_inc)
        ago_cat = ago.iloc[0]["latest_severity_category"]
        check("Angola currently in High category",
              ago_cat == "High",
              "High", ago_cat)
    else:
        check("Angola row present in inform_country_signals",
              False, 1, len(ago))

# ============================================================
# 13) merged structural_neglect_flag fires for 137 plan-rows
# ============================================================
if "structural_neglect_flag" in ranked_gap_scores.columns:
    n_plan_rows_flagged = int(ranked_gap_scores["structural_neglect_flag"].sum())
    check("structural_neglect_flag (v2) fires for 137 plan-rows",
          n_plan_rows_flagged == 137,
          137, n_plan_rows_flagged,
          note="if 0, merge_temporal_signals.py has not been run")

    # All four Tier 1 plans should be True
    tier1_struct = ranked_gap_scores[
        ranked_gap_scores["gap_tier"] == "Tier 1 (most overlooked)"
    ]["structural_neglect_flag"]
    check("All four Tier 1 plans have structural_neglect_flag = True",
          bool(tier1_struct.all()) and len(tier1_struct) == 4,
          "4/4 True", f"{int(tier1_struct.sum())}/{len(tier1_struct)} True")

# ============================================================
# 14) Operating environment scores for Tier 1 + key cases
# ============================================================
expected_op_env = {
    ("LBN", "RSYR25"):  8.5,
    ("COD", "HCOD25"):  9.5,
    ("YEM", "HYEM25"): 10.0,
    ("MMR", "HMMR25b"):10.0,
    ("PSE", "FPSE26"): 10.0,
    ("SDN", "HSDN25"): 10.0,
    ("UKR", "HUKR25"): 10.0,
}
for (iso, code), expected in expected_op_env.items():
    row = ranked_gap_scores[
        (ranked_gap_scores["iso3"] == iso) &
        (ranked_gap_scores["plan_code"] == code)
    ]
    if len(row) == 1:
        actual = row.iloc[0]["operating_env_score"]
        check(f"{iso} {code} operating_env_score = {expected}",
              actual == expected,
              expected, actual)
    else:
        check(f"{iso} {code} row present (for op_env check)",
              False, 1, len(row))

# ============================================================
# 15) INFORM scale jump 1.98x at Dec 2025 -> Jan 2026
# ============================================================
try:
    _ts_available = True
    _ = inform_timeseries
except NameError:
    _ts_available = False
    check("inform_timeseries in scope",
          False, "available",
          "MISSING — run inform_timeseries_v1.py first")

if _ts_available:
    medians = (inform_timeseries
        .groupby(["year", "month"], as_index=False)["severity_score"]
        .median())
    dec_2025 = medians[(medians["year"] == 2025) & (medians["month"] == 12)]
    jan_2026 = medians[(medians["year"] == 2026) & (medians["month"] == 1)]
    if len(dec_2025) == 1 and len(jan_2026) == 1:
        ratio = jan_2026.iloc[0]["severity_score"] / dec_2025.iloc[0]["severity_score"]
        check("INFORM scale jump Dec2025->Jan2026 ratio ~ 1.98x",
              abs(ratio - 1.98) < 0.05,
              "~1.98", f"{ratio:.2f}")
    else:
        check("Dec 2025 and Jan 2026 medians present",
              False, "both", f"dec={len(dec_2025)}, jan={len(jan_2026)}")

# ============================================================
# 16) Durable output files exist
# ============================================================
try:
    current_user = spark.sql("SELECT current_user()").collect()[0][0]
    workspace_dir = f"/Workspace/Users/{current_user}/asli_gulcur_submission"
    expected_files = [
        "gap_score_v1_output.txt",
        "gap_score_v1_ranked.parquet",
        "gap_score_v1_country_rollup.parquet",
        "gap_score_v1_no_plan.parquet",
        "gap_score_v1_sensitivity.json",
        "inform_timeseries_output.txt",
        "inform_timeseries.parquet",
        "inform_country_signals.parquet",
    ]
    for fname in expected_files:
        path = f"{workspace_dir}/{fname}"
        exists = os.path.exists(path)
        check(f"durable output exists: {fname}",
              exists,
              "exists", "exists" if exists else "MISSING", note=path)
except Exception as e:
    check("durable storage check", False, "no error", str(e))

# ============================================================
# Print report
# ============================================================
print("=" * 90)
print(f"VERIFICATION REPORT  —  {len(results)} checks")
print("=" * 90)
pass_count = sum(1 for r in results if r[0] == "PASS")
fail_count = sum(1 for r in results if r[0] == "FAIL")

for status, label, expected, actual, note in results:
    marker = "✓" if status == "PASS" else "✗"
    extra  = f"  ({note})" if note else ""
    print(f"  {marker} [{status}]  {label}")
    if status == "FAIL":
        print(f"           expected: {expected}")
        print(f"           actual:   {actual}{extra}")

print()
print("=" * 90)
print(f"  PASSED: {pass_count} / {len(results)}    FAILED: {fail_count}")
print("=" * 90)
if fail_count == 0:
    print("\n  ✓ ALL CHECKS PASSED — pipeline output matches every documented finding.")
else:
    print(f"\n  ✗ {fail_count} CHECK(S) FAILED — review the actual values above")
    print("    and decide if the writeup needs an update OR the pipeline does.")


VERIFICATION REPORT  —  53 checks
  ✓ [PASS]  crises rows
  ✓ [PASS]  crises columns
  ✓ [PASS]  inform_country rows
  ✓ [PASS]  ranked_gap_scores rows
  ✓ [PASS]  country_rollup rows
  ✓ [PASS]  documented_need_no_plan rows
  ✓ [PASS]  Tier 1 count = 4
  ✓ [PASS]  Tier 2 count = 22
  ✓ [PASS]  Tier 3 count = 46
  ✓ [PASS]  Tier 4 count = 167
  ✓ [PASS]  Tier 1 = exactly 4 plans
  ✓ [PASS]  LBN RSYR25 (Syria 3RP) in Tier 1
  ✓ [PASS]  COD HCOD25 (DRC) in Tier 1
  ✓ [PASS]  YEM HYEM25 (Yemen) in Tier 1
  ✓ [PASS]  MMR HMMR25b (Myanmar) in Tier 1
  ✓ [PASS]  #1 plan is LBN RSYR25
  ✓ [PASS]  Sudan HSDN25 severity = 9.6 (highest in dataset)
  ✓ [PASS]  Sudan HSDN25 ranks Tier 2 (not Tier 1)
  ✓ [PASS]  Palestine 2026 absolute gap ≈ $3.58B
  ✓ [PASS]  Palestine 2026 op_env = 10.0
  ✓ [PASS]  Palestine 2026 is #1 by absolute funding gap globally
  ✓ [PASS]  Palestine uses FTS_requirements_proxy (no HNO PIN)
  ✓ [PASS]  Mauritania in documented_need_no_plan
  ✓ [PASS]  Mauritania severity ≈ 

## What this notebook produced

After a successful Run All:

| Artifact | Form | Where |
|---|---|---|
| `crises` | DataFrame (in-memory) | Stage 1 output |
| `inform_country` | DataFrame (in-memory) | Stage 2 output |
| `ranked_gap_scores`, `country_rollup`, `documented_need_no_plan` | DataFrames + Parquet | Stage 3 + 4 (signals merged in 4) |
| `inform_timeseries`, `inform_country_signals` | DataFrames + Parquet | Stage 4 |
| `sensitivity_report` | dict + JSON | Stage 3 |
| `gap_map.html`, `gap_map.png` | Interactive + static map | Stage 5 |
| Human-readable summary logs | Text logs | Stage 3 + 4 |

## Verifying the output

Open `verification_test.py` from the companion file list and paste it as a final cell. Expected result: **53 / 53 PASS**.

The test checks every documented finding in WRITEUP.md and FINDINGS.md against the actual pipeline output — shape claims, tier distribution, Tier 1 set, Sudan paradox, Palestine multi-lens facts, AFG trajectory, hidden sector gaps, all bonus-task signals, operating-env scores, INFORM scale-jump, and durable-output presence. If the pipeline regresses, the test fails. If the writeup adds an unverifiable number, the test fails. This is the audit trail behind the "data-driven honesty" the writeup commits to.

## Reading further

- **WRITEUP.md** — full technical write-up addressing each PDF criterion, with executive summary on top for the 30-second reader
- **FINDINGS.md** — Good/Bad/Ugly executive summary
- **GAP_SCORE_DESIGN.md** — design rationale with two rounds of assessor self-review (the v1.0 → v1.1 plan-type-aware patch lesson is here)
- **PDF_ALIGNMENT_ASSESSMENT.md** — section-by-section alignment audit with the challenge PDF

## Honest acknowledgments (the limits we name rather than hide)

- **FTS funding is voluntarily self-reported** — donors typically report 3–9 months after disbursement, and non-OECD donors (China, Gulf states) systematically underreport. So `fts_funding` is a **lower bound**, not truth. 2025 coverage rates will revise upward as donor reports settle.
- **HNO 2024 used as fallback for SYR/ETH 2025 need** — disclosed via `hno_source_year` column.
- **Gaza and Lebanon have no HNO PIN** — use FTS requirements as need proxy, labeled `need_source = 'FTS_requirements_proxy'`.
- **`structural_neglect_flag` v1 never fired in our 2-year FTS window** — the original 2-prior-years-below-50%-coverage rule was conservatively designed but couldn't fire on a 2-year history. Replaced in v2 with the scale-invariant INFORM-categorical rule (≥18 of last 24 months at High+), which fires for 137 plan-rows and all four Tier 1 plans.
- **External data enrichment (ACLED / IPC / UNHCR)** — considered, deferred to v2 per PDF Rule 1 ("use the provided datasets as the primary source of truth"). INFORM already incorporates displacement and conflict drivers as sub-dimensions, partially compensating.
- **Sensitivity test results are honest, not flattering** — all four scenarios fail the ≥8/10 top-10 ordinal threshold; the Tier 1 *set* is more stable (3-of-4 always preserved under worst case, fully preserved under severity-only noise). We disclose both numbers.

This system is designed for **decision support, not automated decision-making**. Every score is auditable; every limitation is named.
